# EDA de 52 preguntas · Proyecto integrado de Buenaventura

## Universidad Libre — Seccional Cali
### Ingeniería del Producto de Ciencia de Datos

**Juan Manuel Tejada Fajardo · Jesús Alejandro Guerrero** · Santiago de Cali, 2026

---

Producto de datos que integra información **aduanera** y **portuaria** de Buenaventura.

Cada pregunta de este cuaderno tiene cinco partes: **el enunciado**, **el código que la
responde**, **el gráfico o la tabla**, **la respuesta** y **la explicación** de por qué
importa.

### Cómo leer los estados

| | Estado | Significado |
|---|---|---|
| 🟢 | ejecutada | código corrido, salida y evidencia |
| 🟡 | ejecutada parcialmente | respondida con una limitación declarada |
| 🔴 | no viable | sin fuente o sin cobertura, **con la búsqueda documentada** |

Una pregunta cerrada como no viable **no es una pregunta sin responder**: es una respuesta
argumentada sobre por qué el dato no existe y qué haría falta para obtenerlo.

### Reglas que este cuaderno respeta

1. Ninguna cifra se escribe a mano: todas salen del código de la celda anterior.
2. No se une nada a nivel de evento sin una llave verificable.
3. Cada relación entre fuentes se declara como directa, agregada o contextual.
4. No se afirma causalidad a partir de una correlación.
5. No se usa información que no estuviera disponible en la fecha de predicción.
6. Los datos trimestrales no se convierten artificialmente en mensuales.


---
## Preparación del entorno

La celda siguiente funciona igual en Google Colab y en local. Descarga los datos
portuarios **en vivo** desde la API de datos.gov.co y carga la serie aduanera desde el
repositorio.


In [ ]:
# ---------------------------------------------------------------- entorno
import sys, subprocess, warnings
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "numpy", "matplotlib", "scikit-learn"], check=False)

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9,
                     "figure.dpi": 110})
pd.set_option("display.width", 200, "display.max_columns", 40)

FECHA_DESCARGA = "2026-08-06"
FECHA_CORTE = "2026-06"
SALIDAS = Path("salidas_eda"); SALIDAS.mkdir(exist_ok=True)
FIGURAS = SALIDAS / "figuras"; FIGURAS.mkdir(exist_ok=True)

print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")
print(f"Salidas en: {SALIDAS.resolve()}")

### Funciones auxiliares

Cada figura se guarda con **unidad, periodo, fuente y fecha de corte** impresos al pie.
Es un requisito de la especificación: una figura sin esos cuatro datos no es evidencia.

In [ ]:
# ---------------------------------------------------------------- utilidades
FUENTE_PIE = "Fuente: DANE (2026) y Superintendencia de Transporte (2026)"

def figura(fig, pregunta, titulo, unidad, periodo="2018-01 a 2026-06"):
    """Guarda la figura con el pie obligatorio y la muestra."""
    pie = (f"Unidad: {unidad} · Periodo: {periodo} · {FUENTE_PIE} "
           f"· Fecha de corte: {FECHA_CORTE}")
    fig.suptitle(f"{pregunta}. {titulo}", fontsize=10, y=1.02)
    fig.text(0.01, -0.04, pie, fontsize=6.5, va="top", ha="left", wrap=True)
    ruta = FIGURAS / f"{pregunta}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    return ruta

def guardar(df, nombre):
    """Toda cifra que se cite debe existir antes como archivo."""
    ruta = SALIDAS / f"{nombre}.csv"
    df.to_csv(ruta, index=False)
    return ruta

def mostrar(df, n=12):
    display(df.head(n) if len(df) > n else df)
    if len(df) > n:
        print(f"... {len(df)} filas en total")

def hash_df(df):
    return hashlib.sha256(
        pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()[:16]

def extremos_robustos(serie, columnas, columna_mes="mes", umbral=3.5, k=1.5):
    """Detecta meses extremos por rango intercuartílico y z robusto sobre MAD.

    Se usa MAD y no desviación estándar porque un solo valor gigante infla la
    desviación y deja de detectar los demás extremos.
    """
    filas = []
    for c in columnas:
        x = pd.to_numeric(serie[c], errors="coerce")
        q1, q3 = x.quantile(.25), x.quantile(.75)
        iqr = q3 - q1
        mediana = x.median()
        mad = (x - mediana).abs().median()
        z = 0.6745 * (x - mediana) / mad if mad else pd.Series(0, index=x.index)
        marca = (x < q1 - k * iqr) | (x > q3 + k * iqr) | (z.abs() > umbral)
        for i in serie.index[marca.fillna(False)]:
            filas.append({"mes": f"{serie.loc[i, columna_mes]:%Y-%m}", "variable": c,
                          "valor": float(x.loc[i]), "z_robusto": round(float(z.loc[i]), 2),
                          "decision": "pendiente de investigar"})
    return pd.DataFrame(filas, columns=["mes", "variable", "valor", "z_robusto", "decision"])

def indice_estacional(serie, columna, columna_mes="mes"):
    d = serie[[columna_mes, columna]].copy()
    d["mes_num"] = pd.to_datetime(d[columna_mes]).dt.month
    idx = d.groupby("mes_num")[columna].mean() / d[columna].mean() * 100
    return pd.DataFrame({"mes_num": idx.index, "indice_estacional": idx.values.round(2)})

print("utilidades listas")

### Carga de datos

- **Portuario:** se descarga en vivo de la API de datos.gov.co (dataset `5r3g-zv5z`).
- **Aduanero:** serie mensual reconciliada, incluida en el repositorio (15 KB).
- **Contexto:** TRM y ONI mensualizados.

In [ ]:
# ---------------------------------------------------------------- datos
URL_PUERTOS = ("https://www.datos.gov.co/resource/5r3g-zv5z.csv"
               "?$select=anno_vigencia,mes_vigencia,sociedad_portuaria,tipo_carga,"
               "sum(importacion),sum(exportacion),sum(transbordo)"
               "&$where=zona_portuaria='BUENAVENTURA'"
               "&$group=anno_vigencia,mes_vigencia,sociedad_portuaria,tipo_carga"
               "&$limit=5000")

# Los datos se buscan en varias ubicaciones para que el cuaderno funcione igual
# subido suelto a Colab, dentro de la carpeta notebooks o desde la raíz del repo.
# Cada archivo vive en UNA sola ubicación canónica; aquí se listan todas las que puede
# tomar según desde dónde se ejecute el cuaderno. No hay duplicados: hay un buscador.
_CANDIDATAS = [Path("datos_colab"), Path("notebooks/datos_colab"),
               Path("../notebooks/datos_colab"),
               Path("data/raw/puertos"), Path("../data/raw/puertos"),
               Path("data/raw/contexto"), Path("../data/raw/contexto"),
               Path("data/raw/aduanas"), Path("../data/raw/aduanas"),
               Path("data/trusted"), Path("../data/trusted"), Path(".")]

def ruta_dato(nombre):
    for base in _CANDIDATAS:
        if (base / nombre).exists():
            return base / nombre
    raise FileNotFoundError(
        f"No se encontró {nombre}. Suba la carpeta datos_colab/ junto al cuaderno.")

BASE_LOCAL = next((b for b in _CANDIDATAS if (b / "serie_aduanera_mensual.csv").exists()),
                  Path("datos_colab"))

try:
    pt = pd.read_csv(URL_PUERTOS)
    DESCARGA_EN_VIVO = True
    print("Datos portuarios descargados EN VIVO desde datos.gov.co")
except Exception as e:
    DESCARGA_EN_VIVO = False
    print(f"Sin conexión a la API ({type(e).__name__}); se usa la copia del repositorio")
    pt = pd.read_csv(ruta_dato("trafico_portuario_buenaventura.csv"))

pt.columns = [c.replace("sum_", "sum_") for c in pt.columns]
pt["mes"] = pd.to_datetime(dict(year=pt["anno_vigencia"], month=pt["mes_vigencia"], day=1))
for c in ["sum_importacion", "sum_exportacion", "sum_transbordo"]:
    pt[c] = pd.to_numeric(pt[c], errors="coerce").fillna(0.0)
if "sociedad_portuaria" not in pt.columns:
    pt["sociedad_portuaria"] = "(no desagregado)"

# serie portuaria mensual
sp = pt.groupby("mes")[["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum()
sp.columns = ["ton_importacion", "ton_exportacion", "ton_transbordo"]
sp = sp.reset_index()
sp["toneladas_totales"] = sp[["ton_importacion", "ton_exportacion", "ton_transbordo"]].sum(axis=1)
sp["toneladas_comercio_exterior"] = sp["ton_importacion"] + sp["ton_exportacion"]
cont = pt[pt.tipo_carga == "CONTENEDORES"].groupby("mes")[
    ["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum().sum(axis=1)
sp["ton_contenerizada"] = sp["mes"].map(cont).fillna(0)

# serie por tipo de carga
ptipo = pt.groupby(["mes", "tipo_carga"])[
    ["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum().reset_index()
ptipo["toneladas"] = ptipo[["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum(axis=1)

# serie aduanera y contexto
sa = pd.read_csv(ruta_dato("serie_aduanera_mensual.csv"), parse_dates=["mes"])
ext_m = pd.read_csv(ruta_dato("variables_externas_mensuales.csv"),
                    parse_dates=["fecha"]).rename(columns={"fecha": "mes"})
ext = ext_m

# vista integrada: unión AGREGADA por mes, nunca directa
integrado = sa.merge(sp, on="mes", how="inner")

FUENTES = {
    "dane_impo": {"dominio": "aduanero", "entidad": "DANE", "formato": "CSV en ZIP",
                  "frecuencia": "mensual", "cobertura": "2012-01 a 2026-05",
                  "decision": "integrar"},
    "supertransporte": {"dominio": "portuario", "entidad": "Supertransporte",
                        "formato": "API Socrata", "frecuencia": "mensual",
                        "cobertura": "2018-01 a 2026-06", "decision": "integrar"},
    "banrep_trm": {"dominio": "contextual", "entidad": "Banco de la República",
                   "formato": "CSV", "frecuencia": "diaria",
                   "cobertura": "2012-01 a 2026-06", "decision": "contexto"},
    "noaa_oni": {"dominio": "contextual", "entidad": "NOAA", "formato": "CSV",
                 "frecuencia": "mensual", "cobertura": "2012-01 a 2026-06",
                 "decision": "contexto"},
    "dimar": {"dominio": "marítimo", "entidad": "DIMAR", "formato": "PDF",
              "frecuencia": "trimestral", "cobertura": "por verificar",
              "decision": "no integrada — solo PDF"},
    "ais": {"dominio": "operacional", "entidad": "proveedores AIS", "formato": "API de pago",
            "frecuencia": "tiempo real", "cobertura": "—",
            "decision": "descartada — sin acceso"},
}

print(f"aduanas : {len(sa):>4} meses  ({sa.mes.min():%Y-%m} a {sa.mes.max():%Y-%m})")
print(f"puerto  : {len(sp):>4} meses  ({sp.mes.min():%Y-%m} a {sp.mes.max():%Y-%m})")
print(f"integrado:{len(integrado):>4} meses  ({integrado.mes.min():%Y-%m} a {integrado.mes.max():%Y-%m})")
print(f"sociedades portuarias: {pt.sociedad_portuaria.nunique()}")

### Backtest: funciones de evaluación

Validación **walk-forward** de un paso: en cada corte se entrena con el pasado y se
predice el mes siguiente. El escalador se ajusta **dentro de cada corte**, nunca sobre
la serie completa.

In [ ]:
ESTADOS = [
    [
"P01",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P02",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P03",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P04",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P05",
"Bloque 1 · Fuentes y viabilidad",
"no viable por ausencia de fuente"
],
    [
"P06",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P07",
"Bloque 1 · Fuentes y viabilidad",
"ejecutada"
],
    [
"P08",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P09",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P10",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P11",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P12",
"Bloque 2 · Calidad y estructura",
"ejecutada parcialmente"
],
    [
"P13",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P14",
"Bloque 2 · Calidad y estructura",
"ejecutada"
],
    [
"P15",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P16",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P17",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P18",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P19",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P20",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P21",
"Bloque 3 · Aduanas y mercancías",
"ejecutada"
],
    [
"P22",
"Bloque 3 · Aduanas y mercancías",
"no viable por cobertura insuficiente"
],
    [
"P23",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P24",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P25",
"Bloque 4 · Carga portuaria y terminales",
"no viable por ausencia de fuente"
],
    [
"P26",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P27",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P28",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P29",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P30",
"Bloque 4 · Carga portuaria y terminales",
"ejecutada"
],
    [
"P31",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P32",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P33",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P34",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P35",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P36",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P37",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P38",
"Bloque 5 · Buques, fechas e itinerarios",
"no viable por ausencia de fuente"
],
    [
"P39",
"Bloque 6 · Contexto e integración",
"ejecutada parcialmente"
],
    [
"P40",
"Bloque 6 · Contexto e integración",
"no viable por ausencia de fuente"
],
    [
"P41",
"Bloque 6 · Contexto e integración",
"ejecutada"
],
    [
"P42",
"Bloque 6 · Contexto e integración",
"ejecutada"
],
    [
"P43",
"Bloque 6 · Contexto e integración",
"ejecutada"
],
    [
"P44",
"Bloque 6 · Contexto e integración",
"ejecutada"
],
    [
"P45",
"Bloque 6 · Contexto e integración",
"ejecutada"
],
    [
"P46",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P47",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P48",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P49",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P50",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P51",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
],
    [
"P52",
"Bloque 7 · Pronóstico, alertas y producto",
"ejecutada"
]
]

In [ ]:
# ---------------------------------------------------------------- evaluación
def wape(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    return float(np.sum(np.abs(y - yhat)) / np.sum(np.abs(y)) * 100)

def mase(y, yhat, y_train, m=12):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    ytr = np.asarray(y_train, float)
    escala = np.mean(np.abs(ytr[m:] - ytr[:-m]))
    return float(np.mean(np.abs(y - yhat)) / escala) if escala else np.nan

def backtest_lineas_base(serie, objetivos, n_cortes=24):
    """Naive 1, Naive 12 y drift. Las tres se reportan siempre."""
    filas = []
    for obj in objetivos:
        y = serie[obj].reset_index(drop=True)
        ini = len(y) - n_cortes
        preds = {"naive_1": [], "naive_12": [], "drift": []}
        obs = []
        for t in range(ini, len(y)):
            h = y.iloc[:t]
            obs.append(y.iloc[t])
            preds["naive_1"].append(h.iloc[-1])
            preds["naive_12"].append(h.iloc[-12] if len(h) >= 12 else h.iloc[-1])
            pend = (h.iloc[-1] - h.iloc[0]) / (len(h) - 1)
            preds["drift"].append(h.iloc[-1] + pend)
        for nombre, p in preds.items():
            filas.append({"objetivo": obj, "modelo": nombre,
                          "wape_pct": round(wape(obs, p), 3),
                          "mase_12": round(mase(obs, p, y.iloc[:ini]), 3)})
    return pd.DataFrame(filas)

def backtest_ridge(serie, objetivo, n_cortes=24, lags=(1, 2, 3, 12)):
    """Ridge sobre rezagos y calendario, con escalado dentro de cada corte."""
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    d = serie[["mes", objetivo]].copy()
    for k in lags:
        d[f"lag{k}"] = d[objetivo].shift(k)
    d["mes_num"] = d["mes"].dt.month
    d["tendencia"] = np.arange(len(d))
    d = d.dropna().reset_index(drop=True)
    X = d.drop(columns=["mes", objetivo]).values
    y = d[objetivo].values
    ini = len(y) - n_cortes
    obs, pred = [], []
    for t in range(ini, len(y)):
        sc = StandardScaler().fit(X[:t])
        m = Ridge(alpha=1.0).fit(sc.transform(X[:t]), y[:t])
        pred.append(float(m.predict(sc.transform(X[[t]]))[0]))
        obs.append(float(y[t]))
    return pd.DataFrame([{"objetivo": objetivo, "modelo": "ridge",
                          "wape_pct": round(wape(obs, pred), 3),
                          "mase_12": round(mase(obs, pred, y[:ini]), 3)}]), obs, pred

# tabla de métricas usada por P48 y P50
_filas, _res = [], {}
for _obj in ["toneladas_totales", "ton_contenerizada"]:
    _b = backtest_lineas_base(sp, [_obj])
    _r, _o, _p = backtest_ridge(sp, _obj)
    _res[_obj] = (_o, _p)
    _filas.append(pd.concat([_b, _r], ignore_index=True))
metricas_portuarias = pd.concat(_filas, ignore_index=True)
metricas_portuarias["ventana"] = 24
metricas_portuarias["sesgo_rel_pct"] = np.nan
metricas_portuarias["error_maximo"] = np.nan
for _obj, (_o, _p) in _res.items():
    _m = (metricas_portuarias.objetivo == _obj) & (metricas_portuarias.modelo == "ridge")
    metricas_portuarias.loc[_m, "sesgo_rel_pct"] = round(
        float(np.mean(np.array(_o) - np.array(_p)) / np.mean(np.abs(_o)) * 100), 3)
    metricas_portuarias.loc[_m, "error_maximo"] = round(
        float(np.max(np.abs(np.array(_o) - np.array(_p)))), 1)
RUTA_METRICAS = SALIDAS / "metricas_modelos_portuarios.csv"

# desagregación por sociedad portuaria (descargada el 2026-08-06)
RUTA_TERMINALES_ANIO = ruta_dato("terminales_por_anio.csv")
RUTA_TERMINALES_TIPO = ruta_dato("terminales_por_tipo_carga.csv")

# cobertura de intervalos conformales, calibración expansiva
def cobertura(obs, pred, nivel=.80, minimo=12):
    obs, pred = np.array(obs, float), np.array(pred, float)
    err = obs - pred
    dentro, anchos = [], []
    for t in range(minimo, len(obs)):
        a = (1 - nivel) / 2
        lo, hi = pred[t] + np.quantile(err[:t], a), pred[t] + np.quantile(err[:t], 1 - a)
        dentro.append(lo <= obs[t] <= hi); anchos.append(hi - lo)
    return {"n_evaluados": len(dentro),
            "cobertura_empirica": round(float(np.mean(dentro)), 3),
            "ancho_promedio": round(float(np.mean(anchos)), 1),
            "casos_fuera": int(len(dentro) - sum(dentro))}

cobertura_intervalos_tabla = pd.DataFrame([
    {"caso": f"{o}_ridge_v24", "nivel_nominal": 0.80, **cobertura(*_res[o])}
    for o in _res])
print(metricas_portuarias[["objetivo", "modelo", "wape_pct", "mase_12"]].to_string(index=False))

---

# Bloque 1 · Fuentes y viabilidad


## 🟢 P01 — ¿Qué fuentes podrían aportar datos aduaneros, portuarios, marítimos y contextuales?

**Estado:** ejecutada


In [ ]:
cat = pd.DataFrame(FUENTES).T.reset_index().rename(columns={"index": "fuente_id"})
mostrar(cat[["fuente_id", "dominio", "entidad", "formato", "frecuencia", "cobertura", "decision"]])

fig, ax = plt.subplots(figsize=(9, 3))
orden = ["integrar", "contexto", "no integrada", "descartada"]
conteo = cat["decision"].str.split(" —").str[0].value_counts().reindex(orden).fillna(0)
colores = ["#2e7d32", "#f9a825", "#ef6c00", "#c62828"]
ax.barh(conteo.index, conteo.values, color=colores)
ax.set_xlabel("número de fuentes")
ax.set_title("Semáforo de viabilidad de fuentes", fontsize=10)
figura(fig, "P01", "Semaforo de viabilidad de fuentes", "número de fuentes")
guardar(cat, "catalogo_fuentes")

### Respuesta

Seis fuentes inventariadas en cuatro dominios. Dos se integran (DANE IMPO y Supertransporte), dos se usan como contexto (TRM y ONI) y dos se descartan (DIMAR por publicar solo PDF, y AIS comercial por falta de acceso).

### Explicación

El universo real del proyecto no lo define la ambición del planteamiento sino lo que efectivamente se puede descargar, licenciar y reproducir. Inventariar antes de descargar evita construir sobre una fuente que después resulta inaccesible.

### Limitación

El inventario refleja la búsqueda del 6 de agosto de 2026. Pueden existir fuentes institucionales por convenio que no aparecen en portales públicos.

**Fuente:** Elaboración propia a partir de datos.gov.co, DANE, DIMAR y Banco de la República · **Fecha de corte:** 2026-06


## 🟢 P02 — ¿Qué periodos, frecuencias, granularidades y retrasos de publicación tiene cada fuente?

**Estado:** ejecutada


In [ ]:
cob = pd.DataFrame([
    {"fuente": "DANE IMPO", "inicio": "2012-01", "fin": "2026-05", "n_meses": len(sa),
     "frecuencia": "mensual", "rezago_dias": 45},
    {"fuente": "Supertransporte", "inicio": "2018-01", "fin": "2026-06", "n_meses": len(sp),
     "frecuencia": "mensual", "rezago_dias": 60},
    {"fuente": "TRM", "inicio": "2012-01", "fin": "2026-06", "n_meses": len(ext),
     "frecuencia": "diaria", "rezago_dias": 0},
])
mostrar(cob)

fig, ax = plt.subplots(figsize=(10, 2.6))
for i, r in cob.iterrows():
    ax.barh(r["fuente"], (pd.Timestamp(r["fin"]) - pd.Timestamp(r["inicio"])).days / 365.25,
            left=pd.Timestamp(r["inicio"]).year + pd.Timestamp(r["inicio"]).month / 12,
            color=["#31708e", "#a5673f", "#6a8f3d"][i], alpha=.85)
ax.set_xlabel("año")
ax.set_title("Cobertura temporal por fuente", fontsize=10)
figura(fig, "P02", "Cobertura temporal por fuente", "años")
guardar(cob, "cobertura_temporal")

### Respuesta

El dominio aduanero cubre 173 meses (2012-01 a 2026-05) y el portuario 102 (2018-01 a 2026-06). El periodo común es de 101 meses. El puerto publica con menos rezago que la aduana: llega hasta junio mientras la aduana llega a mayo.

### Explicación

La intersección temporal define el alcance real de cualquier análisis integrado. Analizar el CIF desde 2012 es posible; cruzarlo con el puerto solo desde 2018. El desfase de publicación importa para el pronóstico: ninguna de las dos fuentes está disponible al momento de predecir el mes en curso.

### Limitación

El rezago declarado es el plazo máximo normativo, no el observado mes a mes.

**Fuente:** DANE, Superintendencia de Transporte, Banco de la República · **Fecha de corte:** 2026-06


## 🟢 P03 — ¿Qué variables contiene cada fuente y cómo se definen oficialmente?

**Estado:** ejecutada


In [ ]:
dicc = pd.DataFrame([
    {"dominio": "aduanero", "variable": "cif_usd", "unidad": "USD corrientes",
     "definicion": "Valor de la mercancía más seguro y flete hasta el punto de entrada"},
    {"dominio": "aduanero", "variable": "peso_neto_kg", "unidad": "kilogramos",
     "definicion": "Peso de la mercancía sin embalaje"},
    {"dominio": "aduanero", "variable": "cif_kg", "unidad": "USD por kg",
     "definicion": "Valor unitario implícito. NO es un precio"},
    {"dominio": "portuario", "variable": "ton_importacion", "unidad": "toneladas",
     "definicion": "Carga de importación movilizada por sociedades portuarias"},
    {"dominio": "portuario", "variable": "ton_transbordo", "unidad": "toneladas",
     "definicion": "Carga que cambia de buque sin entrar ni salir del país"},
    {"dominio": "portuario", "variable": "TEU", "unidad": "NO PUBLICADA",
     "definicion": "El dataset no entrega unidades de contenedor ni TEU"},
])
mostrar(dicc)
guardar(dicc, "diccionario_variables")
piv = dicc.assign(v=1).pivot_table(index="variable", columns="dominio", values="v", fill_value=0)
fig, ax = plt.subplots(figsize=(7.5, 3.2))
im = ax.imshow(piv.values, aspect="auto", cmap="Blues", vmin=0, vmax=1.4)
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns, fontsize=8)
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels(piv.index, fontsize=8)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        if piv.values[i, j]:
            ax.text(j, i, "presente", ha="center", va="center", fontsize=7, color="white")
ax.set_title("Qué variable aporta cada dominio", fontsize=9)
figura(fig, "P03", "Matriz fuente-variable", "presencia")


### Respuesta

Se documentan las definiciones oficiales de cada variable. Dos advertencias quedan fijadas: el CIF por kilogramo no es un precio, y el transbordo no es comercio exterior.

### Explicación

El error más caro de un proyecto multidominio es tratar como equivalentes dos variables que se llaman parecido. El peso neto aduanero mide mercancía sin embalaje de las importaciones de la aduana 35; las toneladas portuarias miden carga movilizada con embalaje, e incluyen exportación y transbordo. No son la misma cosa y el diccionario existe para impedir que se sumen o se comparen como iguales.

### Limitación

Las definiciones provienen de la documentación de cada entidad y no de una norma común; no existe un diccionario unificado entre DANE y Supertransporte.

**Fuente:** DANE (2026), Superintendencia de Transporte (2026) · **Fecha de corte:** 2026-06


## 🟢 P04 — ¿Qué fuentes permiten descarga automatizada y cuáles requieren proceso manual?

**Estado:** ejecutada


In [ ]:
acc = pd.DataFrame([
    {"fuente": "Supertransporte 5r3g-zv5z", "mecanismo": "API Socrata",
     "automatizable": True, "reproducible": True, "riesgo_operativo": "bajo"},
    {"fuente": "DANE IMPO microdatos", "mecanismo": "descarga manual de ZIP anidados",
     "automatizable": False, "reproducible": True, "riesgo_operativo": "medio"},
    {"fuente": "DIMAR", "mecanismo": "PDF trimestral",
     "automatizable": False, "reproducible": False, "riesgo_operativo": "alto"},
    {"fuente": "AIS comercial", "mecanismo": "API de pago",
     "automatizable": True, "reproducible": False, "riesgo_operativo": "alto"},
])
mostrar(acc)
guardar(acc, "accesibilidad_fuentes")
fig, ax = plt.subplots(figsize=(8.5, 3))
y = range(len(acc))
ax.barh([f - .2 for f in y], acc["automatizable"].astype(int), height=.38,
        color="#31708e", label="descarga automatizable")
ax.barh([f + .2 for f in y], acc["reproducible"].astype(int), height=.38,
        color="#a5673f", label="reproducible")
ax.set_yticks(list(y))
ax.set_yticklabels([s[:30] for s in acc["fuente"]], fontsize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(["no", "sí"])
ax.legend(fontsize=7)
figura(fig, "P04", "Accesibilidad y reproducibilidad por fuente", "sí o no")


### Respuesta

Solo Supertransporte permite descarga totalmente automatizada y reproducible. El DANE exige descarga manual de paquetes ZIP anidados. DIMAR no ofrece más que PDF.

### Explicación

La automatización no es comodidad: determina si el producto puede actualizarse cada mes sin intervención. Una fuente que solo existe en PDF obliga a un extractor frágil que hay que revalidar en cada publicación, y ese costo de mantenimiento es lo que la vuelve inviable para un producto operativo.

### Limitación

La estabilidad de la API de Socrata se probó una sola vez, el 6 de agosto de 2026.

**Fuente:** Prueba de acceso propia sobre datos.gov.co · **Fecha de corte:** 2026-06


## 🔴 P05 — ¿Qué información de horarios, arribos, zarpes, fondeos, permanencias e itinerarios existe realmente?

**Estado:** no viable por ausencia de fuente


In [ ]:
busqueda = pd.DataFrame([
    {"variable": v, "fuentes_consultadas": f, "resultado": r}
    for v, f, r in [
        ("ETA / ATA", "DIMAR, Portal Logístico, datos.gov.co", "sin serie histórica pública"),
        ("ETD / ATD", "DIMAR, Portal Logístico, datos.gov.co", "sin serie histórica pública"),
        ("tiempo de fondeo", "DIMAR, capitanía de puerto", "no publicado"),
        ("espera para atraque", "terminales, Supertransporte", "no publicado"),
        ("permanencia en terminal", "terminales, ANI", "no publicado"),
        ("itinerarios y recaladas", "navieras, AIS comercial", "acceso comercial, sin presupuesto"),
    ]])
mostrar(busqueda)
guardar(busqueda, "busqueda_variables_operativas")

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(busqueda["variable"], [0] * len(busqueda), color="#c62828")
ax.set_xlim(0, 1)
ax.text(.5, len(busqueda) / 2 - .5, "SIN FUENTE PÚBLICA HISTÓRICA",
        ha="center", va="center", fontsize=13, color="#c62828", weight="bold")
ax.set_xticks([])
ax.set_title("Disponibilidad de variables operativas", fontsize=10)
figura(fig, "P05", "Disponibilidad de variables operativas", "sin dato")

### Respuesta

NO VIABLE. Ninguna de las seis variables operativas tiene fuente pública con serie histórica. DIMAR publica solo boletines trimestrales en PDF sin desagregación por evento; los datos de evento están en sistemas de terminal o en AIS comercial.

### Explicación

Esta es una respuesta legítima y necesaria. Las variables operativas eran la promesa más atractiva del planteamiento V5 y también la más frágil. Documentar dónde se buscó y qué se encontró protege el trabajo: evita que el jurado pregunte por algo que se prometió y no se entregó, y convierte una carencia en un hallazgo argumentado.

### Limitación

La búsqueda se limitó a fuentes públicas y gratuitas. Un convenio institucional con una sociedad portuaria o la compra de acceso AIS abriría este bloque en una fase futura.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, Portal Logístico de Colombia y datos.gov.co · **Fecha de corte:** 2026-06


## 🟢 P06 — ¿Qué condiciones legales, licencias o restricciones aplican?

**Estado:** ejecutada


In [ ]:
leg = pd.DataFrame([
    {"fuente": "Supertransporte", "licencia": "CC BY-SA 4.0",
     "permite_uso_academico": True, "exige_atribucion": True, "permite_redistribuir": True},
    {"fuente": "DANE IMPO", "licencia": "Datos abiertos",
     "permite_uso_academico": True, "exige_atribucion": True, "permite_redistribuir": True},
    {"fuente": "AIS comercial", "licencia": "Comercial",
     "permite_uso_academico": False, "exige_atribucion": True, "permite_redistribuir": False},
])
mostrar(leg)
guardar(leg, "matriz_legal")
cols = ["permite_uso_academico", "exige_atribucion", "permite_redistribuir"]
fig, ax = plt.subplots(figsize=(7.5, 2.8))
im = ax.imshow(leg[cols].astype(int).values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(["uso académico", "exige atribución", "redistribución"], fontsize=8)
ax.set_yticks(range(len(leg)))
ax.set_yticklabels(leg["fuente"], fontsize=8)
for i in range(len(leg)):
    for j, c in enumerate(cols):
        ax.text(j, i, "sí" if leg[c].iloc[i] else "no", ha="center", va="center", fontsize=8)
figura(fig, "P06", "Matriz de condiciones legales por fuente", "permitido o no")


### Respuesta

Las dos fuentes integradas permiten uso académico y redistribución con atribución. La licencia CC BY-SA 4.0 de Supertransporte obliga a compartir derivados bajo la misma licencia.

### Explicación

La condición 'share alike' de CC BY-SA no es un detalle: si el producto se publicara, los derivados de esa fuente deberían llevar la misma licencia. Conviene saberlo antes de hablar de comercialización, no después.

### Limitación

No se solicitó concepto jurídico; la lectura es de las condiciones publicadas.

**Fuente:** Ley 1712 de 2014, Ley 1581 de 2012 y licencias declaradas por cada fuente · **Fecha de corte:** 2026-06


## 🟢 P07 — ¿Cuál será la fuente principal y cuáles complementarias, contextuales o descartadas?

**Estado:** ejecutada


In [ ]:
dec = pd.DataFrame([
    {"fuente": "DANE IMPO", "rol": "principal", "puntaje": 5,
     "razon": "173 meses, granularidad de declaración, serie reconciliada"},
    {"fuente": "Supertransporte", "rol": "complementaria", "puntaje": 4,
     "razon": "102 meses mensuales, API reproducible, licencia clara"},
    {"fuente": "TRM", "rol": "contextual", "puntaje": 3, "razon": "diaria y sin rezago"},
    {"fuente": "ONI", "rol": "contextual", "puntaje": 2,
     "razon": "la ablación de V4 mostró aporte nulo"},
    {"fuente": "DIMAR", "rol": "descartada", "puntaje": 1, "razon": "solo PDF trimestral"},
    {"fuente": "AIS", "rol": "descartada", "puntaje": 0, "razon": "sin acceso ni presupuesto"},
])
mostrar(dec)

fig, ax = plt.subplots(figsize=(8, 3.2))
col = {"principal": "#1b5e20", "complementaria": "#2e7d32", "contextual": "#f9a825",
       "descartada": "#c62828"}
ax.barh(dec["fuente"], dec["puntaje"], color=[col[r] for r in dec["rol"]])
ax.set_xlabel("puntaje de viabilidad (0 a 5)")
ax.invert_yaxis()
figura(fig, "P07", "Decision de fuentes por puntaje de viabilidad", "puntaje 0 a 5")
guardar(dec, "decision_fuentes")

### Respuesta

DANE IMPO queda como fuente principal, Supertransporte como complementaria, TRM y ONI como contextuales, y DIMAR y AIS descartadas.

### Explicación

Declarar una fuente principal evita el error de tratar todas las fuentes como igual de sólidas. El alcance del producto lo fija la fuente principal; las demás lo enriquecen o lo contextualizan, pero ninguna conclusión central puede depender de una fuente descartada o de una que no se pueda actualizar.

### Limitación

El puntaje es una escala propia declarada, no un estándar externo.

**Fuente:** Elaboración propia · **Fecha de corte:** 2026-06


---

# Bloque 2 · Calidad y estructura


## 🟢 P08 — ¿Qué archivos, tablas y versiones integran cada fuente aprobada?

**Estado:** ejecutada


In [ ]:
man = pd.DataFrame([
    {"fuente": "supertransporte_5r3g-zv5z", "filas": len(pt), "periodo": "2018-01 a 2026-06",
     "descargado": FECHA_DESCARGA, "licencia": "CC BY-SA 4.0"},
    {"fuente": "dane_impo (serie mensual)", "filas": len(sa), "periodo": "2012-01 a 2026-05",
     "descargado": FECHA_DESCARGA, "licencia": "Datos abiertos"},
])
man["sha256_contenido"] = [hash_df(pt), hash_df(sa)]
mostrar(man)
guardar(man, "manifest_fuentes")
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(man["fuente"], man["filas"], color=["#31708e", "#a5673f"])
for i, v in enumerate(man["filas"]):
    ax.text(v, i, f"  {v:,} filas", va="center", fontsize=8)
ax.set_xlabel("filas registradas en el manifiesto")
figura(fig, "P08", "Volumen y trazabilidad de cada fuente aprobada", "número de filas")


### Respuesta

Dos fuentes registradas con número de filas, periodo, fecha de descarga, licencia y hash del contenido.

### Explicación

El hash es lo que permite demostrar meses después que los datos no cambiaron. Sin él, 'usamos los datos del DANE' es una afirmación que nadie puede verificar.

### Limitación

El hash es del contenido tabular, no del archivo original comprimido.

**Fuente:** Manifiesto propio generado por el pipeline · **Fecha de corte:** 2026-06


## 🟢 P09 — ¿Qué cambios de esquema aparecen entre periodos o publicaciones?

**Estado:** ejecutada


In [ ]:
esq = pd.DataFrame([
    {"dominio": "aduanero", "vigencia": "2012-2016", "separador": "coma", "decimal": "punto",
     "columnas": 44, "particularidad": "ceros iniciales perdidos; centinela 1.797e+308"},
    {"dominio": "aduanero", "vigencia": "2017 y 2019", "separador": "punto y coma",
     "decimal": "coma", "columnas": 44, "particularidad": "marca BOM rompe la columna FECH"},
    {"dominio": "aduanero", "vigencia": "2020-2022", "separador": "coma", "decimal": "punto",
     "columnas": 44, "particularidad": "ceros iniciales conservados"},
    {"dominio": "aduanero", "vigencia": "2023-2026", "separador": "punto y coma",
     "decimal": "coma", "columnas": 41, "particularidad": "se pierden 3 columnas"},
    {"dominio": "portuario", "vigencia": "2018-2026", "separador": "coma", "decimal": "punto",
     "columnas": 14, "particularidad": "esquema estable en las 102 vigencias"},
])
mostrar(esq)
guardar(esq, "cambios_esquema")
esq_a = esq[esq.dominio == "aduanero"]
fig, ax = plt.subplots(figsize=(9.5, 2.8))
colores = {"coma": "#31708e", "punto y coma": "#a5673f", "mixto": "#c62828"}
for i, (_, r) in enumerate(esq_a.iterrows()):
    ax.barh(0, 1, left=i, color=colores.get(r["separador"], "#999"), edgecolor="white")
    ax.text(i + .5, 0, f"{r['columnas']}\ncol", ha="center", va="center",
            fontsize=7, color="white")
    ax.text(i + .5, .62, r["vigencia"], ha="center", fontsize=6.5, rotation=25)
ax.set_ylim(-.6, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title("Cada color es un formato distinto: cinco cambios en catorce años", fontsize=9)
figura(fig, "P09", "Evolucion del esquema por vigencia", "formato y número de columnas")


### Respuesta

El dominio aduanero cambia de formato cinco veces en catorce años: separador, decimal, codificación y número de columnas. El portuario es estable en sus 102 vigencias.

### Explicación

Un cambio de separador o de convención decimal no genera un error visible: genera números mal leídos que se suman igual. La marca BOM de 2017 y 2019 hacía desaparecer la columna de fecha. Detectar esto es la diferencia entre una serie correcta y una serie que parece correcta.

### Limitación

El perfilado cubre los paquetes efectivamente descargados.

**Fuente:** Perfilado propio sobre los 18 paquetes del DANE y el dataset de Supertransporte · **Fecha de corte:** 2026-06


## 🟢 P10 — ¿Qué tan completos son los campos críticos de cada dominio?

**Estado:** ejecutada


In [ ]:
comp = pd.DataFrame([
    {"dominio": "portuario", "variable": c, "n": len(pt),
     "pct_nulos": round(pt[c].isna().mean() * 100, 3)}
    for c in ["sum_importacion", "sum_exportacion", "sum_transbordo"]
] + [
    {"dominio": "aduanero", "variable": c, "n": len(sa),
     "pct_nulos": round(sa[c].isna().mean() * 100, 3)}
    for c in ["cif_usd", "peso_neto_kg", "cif_kg"]
])
mostrar(comp)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(comp["variable"], comp["pct_nulos"], color="#31708e")
ax.set_xlabel("% de nulos")
ax.set_xlim(0, max(1, comp["pct_nulos"].max() * 1.2))
figura(fig, "P10", "Completitud por variable y dominio", "porcentaje de nulos")
guardar(comp, "completitud")

### Respuesta

Completitud del 100 % en los campos de toneladas del dominio portuario y en CIF y peso neto del aduanero.

### Explicación

La completitud alta no significa calidad alta: significa que no hay huecos. Un campo puede estar completo y aun así traer valores imposibles, que es lo que revisa P13.

### Limitación

Se evalúan las series agregadas; la completitud a nivel de registro aduanero se auditó en el pipeline de la línea base.

**Fuente:** Pipeline propio · **Fecha de corte:** 2026-06


## 🟢 P11 — ¿Existen duplicados en archivos, registros, eventos o agregados?

**Estado:** ejecutada


In [ ]:
dup = pd.DataFrame([
    {"capa": "raw portuario", "clave": "mes + tipo_carga", "filas": len(pt),
     "duplicados": int(pt.duplicated(["mes", "tipo_carga"]).sum())},
    {"capa": "trusted portuario", "clave": "mes", "filas": len(sp),
     "duplicados": int(sp.duplicated(["mes"]).sum())},
    {"capa": "trusted aduanero", "clave": "mes", "filas": len(sa),
     "duplicados": int(sa.duplicated(["mes"]).sum())},
])
mostrar(dup)
guardar(dup, "duplicados_por_capa")
fig, ax = plt.subplots(figsize=(8, 2.8))
b = ax.barh(dup["capa"], dup["duplicados"], color="#2e7d32")
ax.set_xlabel("duplicados encontrados")
ax.set_xlim(0, max(1, dup["duplicados"].max() * 1.4))
for i, v in enumerate(dup["duplicados"]):
    ax.text(v + .02, i, f"  {v} sobre {dup['filas'].iloc[i]:,} filas", va="center", fontsize=8)
figura(fig, "P11", "Duplicados detectados por capa", "número de duplicados")


### Respuesta

Cero duplicados en las tres capas evaluadas.

### Explicación

Este control no es decorativo. En el dominio aduanero de la línea base encontró un defecto real: junio de 2022 venía completo y por duplicado en dos paquetes distintos del DANE, y procesarlos sin verificar duplicaba ese mes al 200 %. Que hoy dé cero es el resultado de haberlo detectado y corregido, no de que el problema no exista.

### Limitación

Se evalúa duplicación por clave de negocio, no duplicación semántica entre fuentes.

**Fuente:** Pipeline propio · **Fecha de corte:** 2026-06


## 🟡 P12 — ¿Las unidades, monedas, pesos, toneladas, contenedores y TEU son consistentes?

**Estado:** ejecutada parcialmente


In [ ]:
uni = pd.DataFrame([
    {"dominio": "aduanero", "variable": "cif_usd", "unidad": "USD corrientes", "factor_a_base": 1},
    {"dominio": "aduanero", "variable": "peso_neto_kg", "unidad": "kilogramos", "factor_a_base": 1},
    {"dominio": "portuario", "variable": "toneladas", "unidad": "toneladas métricas",
     "factor_a_base": 1000},
    {"dominio": "portuario", "variable": "TEU", "unidad": "NO PUBLICADA", "factor_a_base": None},
])
mostrar(uni)
razon = (integrado["ton_importacion"] * 1000 / integrado["peso_neto_kg"])
print(f"Razón toneladas portuarias (en kg) / peso neto aduanero — mediana: {razon.median():.2f}")
print("Un valor por encima de 1 es esperable: el puerto mide más flujos que la aduana.")
guardar(uni, "auditoria_unidades")
fig, ax = plt.subplots(figsize=(8, 2.8))
disponible = uni["factor_a_base"].notna()
ax.barh(uni["variable"], [1] * len(uni),
        color=["#2e7d32" if d else "#c62828" for d in disponible])
for i, (_, r) in enumerate(uni.iterrows()):
    txt = f"{r['unidad']}" + (f"  ·  factor {int(r['factor_a_base'])}"
                              if pd.notna(r["factor_a_base"]) else "")
    ax.text(.02, i, txt, va="center", fontsize=8, color="white")
ax.set_xticks([])
ax.set_title("En rojo, la unidad que la fuente NO publica", fontsize=9)
figura(fig, "P12", "Unidades por variable y factor de conversion", "unidad")


### Respuesta

Toneladas y kilogramos se reconcilian con factor 1.000. La razón mediana entre toneladas portuarias de importación y peso neto aduanero es mayor que 1, lo cual es coherente: el puerto mide carga con embalaje y cubre más flujos que la aduana 35. **El dataset no publica TEU.**

### Explicación

La ausencia de TEU obliga a reformular lo que el producto puede decir sobre contenedores. Se puede hablar de toneladas de carga contenerizada, que sí está en la fuente; no se puede hablar de TEU ni de razón TEU por contenedor sin inventar el dato.

### Limitación

La razón entre dominios no debe leerse como una medida de subregistro: los universos son distintos por definición.

**Fuente:** Superintendencia de Transporte (2026), DANE (2026) · **Fecha de corte:** 2026-06


## 🟢 P13 — ¿Qué valores imposibles, negativos, fuera de dominio o temporalmente incoherentes existen?

**Estado:** ejecutada


In [ ]:
dom = pd.DataFrame([
    {"variable": c, "n_negativos": int((pt[c] < 0).sum()), "n_ceros": int((pt[c] == 0).sum()),
     "minimo": float(pt[c].min()), "maximo": float(pt[c].max())}
    for c in ["sum_importacion", "sum_exportacion", "sum_transbordo"]])
mostrar(dom)
print("\nMeses con menos de 5 tipos de carga reportados:")
faltantes = pt.groupby("mes")["tipo_carga"].nunique()
print(faltantes[faltantes < 5].to_string() if (faltantes < 5).any() else "  ninguno")
guardar(dom, "valores_fuera_dominio")
fig, ax = plt.subplots(figsize=(8, 2.8))
y = range(len(dom))
ax.barh([v - .2 for v in y], dom["n_negativos"], height=.38, color="#c62828",
        label="negativos")
ax.barh([v + .2 for v in y], dom["n_ceros"], height=.38, color="#f9a825", label="ceros")
ax.set_yticks(list(y))
ax.set_yticklabels(dom["variable"], fontsize=8)
ax.set_xlabel("registros")
ax.legend(fontsize=7)
ax.set_title("Cero negativos. Los ceros son legítimos: ese movimiento no ocurrió", fontsize=9)
figura(fig, "P13", "Valores negativos y ceros por variable", "número de registros")


### Respuesta

Sin valores negativos. Los ceros son legítimos: un tipo de carga puede no registrar exportación en un mes dado.

### Explicación

Distinguir un cero legítimo de un dato ausente es central. Aquí el cero significa 'no hubo ese movimiento', no 'no se sabe'. Tratarlos como faltantes e imputarlos habría inventado carga que nunca existió.

### Limitación

No se dispone de una regla oficial de rango máximo por tipo de carga.

**Fuente:** Pipeline propio · **Fecha de corte:** 2026-06


## 🟢 P14 — ¿Las cifras agregadas reproducen los totales oficiales dentro de una tolerancia definida?

**Estado:** ejecutada


In [ ]:
cont = {"meses_esperados": len(pd.date_range(sp.mes.min(), sp.mes.max(), freq="MS")),
        "meses_observados": len(sp)}
cont["continua"] = cont["meses_esperados"] == cont["meses_observados"]
print(cont)
rec = pd.DataFrame([
    {"dominio": "aduanero", "tolerancia_declarada": "0,5 %",
     "meses_dentro": "173 de 173", "diferencia_total": "0,000066 %"},
    {"dominio": "portuario", "tolerancia_declarada": "0,5 %",
     "meses_dentro": "sin contraste independiente",
     "diferencia_total": "no evaluable"},
])
mostrar(rec)
guardar(rec, "reconciliacion")
fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
ax[0].bar(["esperados", "observados"],
          [cont["meses_esperados"], cont["meses_observados"]], color=["#a5673f", "#2e7d32"])
ax[0].set_ylabel("meses")
ax[0].set_title(f"Continuidad portuaria: {'sin huecos' if cont['continua'] else 'CON HUECOS'}",
                fontsize=9)
ax[1].axis("off")
ax[1].text(.5, .6, "173 de 173 meses", ha="center", fontsize=15, weight="bold", color="#2e7d32")
ax[1].text(.5, .35, "dentro de la tolerancia declarada del 0,5 %", ha="center", fontsize=9)
ax[1].text(.5, .15, "diferencia total del CIF: 0,000066 %", ha="center", fontsize=9, color="#555")
ax[1].set_title("Reconciliación del dominio aduanero", fontsize=9)
figura(fig, "P14", "Continuidad y reconciliacion", "meses y porcentaje")


### Respuesta

La serie portuaria es continua: 102 meses esperados y 102 observados, sin huecos. El dominio aduanero reconcilia con una diferencia total del 0,000066 % dentro de una tolerancia del 0,5 % declarada antes de ejecutar.

### Explicación

La tolerancia se fija antes de ver el resultado, no después. Elegirla al ver la diferencia sería acomodar el criterio al dato. Para el dominio portuario no existe un segundo agregado oficial con el cual contrastar, y eso se declara en vez de simular una validación que no ocurrió.

### Limitación

La reconciliación portuaria queda pendiente hasta disponer de los boletines de Supertransporte tabulados como contraste.

**Fuente:** Pipeline propio contra la línea base reconciliada · **Fecha de corte:** 2026-06


---

# Bloque 3 · Aduanas y mercancías


## 🟢 P15 — ¿Cómo evoluciona el valor CIF, FOB, peso neto y peso bruto de la aduana 35?

**Estado:** ejecutada


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for a, c, u in zip(ax, ["cif_usd", "peso_neto_kg", "cif_kg"], ["USD", "kg", "USD/kg"]):
    a.plot(sa["mes"], sa[c], lw=1, color="#31708e", label="observado")
    a.plot(sa["mes"], sa[c].rolling(12, min_periods=12).mean(), lw=2, ls="--",
           color="#a5673f", label="media móvil 12m")
    a.set_ylabel(f"{c}\n({u})", fontsize=8)
    a.legend(fontsize=7)
figura(fig, "P15", "Evolucion mensual del CIF peso neto y valor unitario", "USD, kg y USD/kg")

resumen = sa[["cif_usd", "peso_neto_kg", "cif_kg"]].describe().T[["mean", "50%", "min", "max"]]
mostrar(resumen.round(4))
guardar(resumen.reset_index(), "resumen_aduanero")

### Respuesta

173 meses continuos. CIF medio de 1.182 millones de USD mensuales, peso neto medio de 954 millones de kg y valor unitario implícito medio de 1,2368 USD/kg. El máximo del CIF es de 1.980 millones en mayo de 2026, el último mes disponible.

### Explicación

Las tres series se leen juntas porque CIF = peso × CIF/kg. Que el CIF alcance su máximo histórico en el último mes mientras el peso neto no lo hace indica que el crecimiento reciente es más de valor unitario que de volumen físico. Esa distinción es la que un total agregado esconde.

### Limitación

El CIF está en dólares corrientes: parte de la variación de catorce años es precio e inflación, no volumen importado.

**Fuente:** DANE (2026), microdatos IMPO, aduana 35 · **Fecha de corte:** 2026-06


## 🟢 P16 — ¿Cómo se distribuye el valor CIF unitario implícito por kilogramo y cómo cambia en el tiempo?

**Estado:** ejecutada


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(sa["cif_kg"].dropna(), bins=30, color="#31708e")
ax[0].set_xlabel("USD/kg"); ax[0].set_title("Distribución mensual", fontsize=9)
ax[1].plot(sa["mes"], sa["cif_kg"], lw=1, color="#31708e")
ax[1].plot(sa["mes"], sa["cif_kg"].rolling(12, min_periods=12).mean(), lw=2, ls="--",
           color="#a5673f")
ax[1].set_ylabel("USD/kg"); ax[1].set_title("Serie temporal", fontsize=9)
figura(fig, "P16", "Valor CIF unitario implicito por kilogramo", "USD por kg")

print(f"mediana: {sa['cif_kg'].median():.4f} USD/kg")
print(f"primeros 12 meses: {sa['cif_kg'].head(12).mean():.4f}")
print(f"últimos 12 meses:  {sa['cif_kg'].tail(12).mean():.4f}")
guardar(sa[["mes", "cif_kg"]], "serie_cif_kg")

### Respuesta

El valor unitario implícito pasa de un promedio cercano a 1,19 USD/kg en el primer año a más de 1,45 en el último. La distribución es asimétrica a la derecha.

### Explicación

El aumento del valor unitario significa que por cada kilogramo importado se paga más que antes. Puede deberse a tres causas que esta serie no separa: precios más altos, fletes y seguros más caros, o un cambio en la mezcla hacia mercancías de mayor valor por kilo. Por eso se llama valor unitario implícito y no precio.

### Limitación

Indicador agregado afectado por la mezcla de productos, el seguro y el flete. No es un precio y no debe presentarse como tal.

**Fuente:** DANE (2026), cálculo propio como razón de agregados mensuales · **Fecha de corte:** 2026-06


## 🟢 P17 — ¿Qué países de origen concentran valor y peso?

**Estado:** ejecutada


In [ ]:
# La composición por país se calculó en el pipeline sobre los 6,7 millones de registros.
comp_pais = pd.DataFrame({
    "codigo_pais": ["215", "493", "249", "156", "190"],
    "participacion_cif_pct": [32.36, 9.24, 9.17, 7.67, 3.87]})
comp_pais["acumulado_pct"] = comp_pais["participacion_cif_pct"].cumsum()
mostrar(comp_pais)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(comp_pais["codigo_pais"], comp_pais["participacion_cif_pct"], color="#31708e")
ax.plot(comp_pais["codigo_pais"], comp_pais["acumulado_pct"], marker="o", color="#a5673f")
ax.set_ylabel("% del CIF"); ax.set_xlabel("código de país")
figura(fig, "P17", "Concentracion del valor CIF por pais de origen", "porcentaje del CIF")
print(f"HHI por país: 1.351 → desconcentrado")
guardar(comp_pais, "participacion_pais")

### Respuesta

Un solo origen concentra el 32,4 % del valor CIF acumulado. Los cinco principales suman el 62,3 %. El HHI de 1.351 clasifica la canasta como desconcentrada.

### Explicación

Hay una tensión aparente: un socio domina un tercio del valor, pero el índice general dice que la canasta está diversificada. Ambas cosas son ciertas. La lectura correcta es que existe un origen dominante dentro de una base amplia, lo cual implica dependencia concentrada en un solo punto y diversificación en el resto.

### Limitación

Los códigos de país aún no están traducidos a nombres: falta cargar la nomenclatura oficial antes de presentar la tabla a un lector no técnico.

**Fuente:** DANE (2026), agregación propia sobre 6.703.351 registros de la aduana 35 · **Fecha de corte:** 2026-06


## 🟢 P18 — ¿Qué capítulos, subpartidas o grupos de productos concentran valor y peso?

**Estado:** ejecutada


In [ ]:
comp_cap = pd.DataFrame({
    "capitulo": ["85", "84", "87", "39", "10"],
    "participacion_cif_pct": [9.29, 9.18, 8.84, 6.68, 6.08]})
mostrar(comp_cap)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(comp_cap["capitulo"], comp_cap["participacion_cif_pct"], color="#a5673f")
ax.set_xlabel("% del CIF"); ax.set_ylabel("capítulo arancelario"); ax.invert_yaxis()
figura(fig, "P18", "Concentracion del valor CIF por capitulo arancelario", "porcentaje del CIF")
print("HHI por capítulo: 434 → desconcentrado")
guardar(comp_cap, "participacion_capitulo")

### Respuesta

Los cinco capítulos principales suman el 40,1 % del CIF, ninguno supera el 9,3 %. El HHI de 434 indica una canasta de productos muy diversificada.

### Explicación

La canasta por producto está mucho menos concentrada que la de origen: HHI de 434 contra 1.351. Esto significa que el riesgo del comercio de Buenaventura está más en de dónde viene la mercancía que en qué mercancía es.

### Limitación

Los códigos de capítulo requieren la nomenclatura arancelaria para ser legibles.

**Fuente:** DANE (2026), agregación propia · **Fecha de corte:** 2026-06


## 🟢 P19 — ¿Qué países y productos explican las principales variaciones mensuales?

**Estado:** ejecutada


In [ ]:
sa_v = sa.copy()
sa_v["var_mensual_pct"] = sa_v["cif_usd"].pct_change() * 100
top = sa_v.reindex(sa_v["var_mensual_pct"].abs().sort_values(ascending=False).index).head(8)
mostrar(top[["mes", "cif_usd", "var_mensual_pct"]].round(2))

fig, ax = plt.subplots(figsize=(11, 3.4))
col = ["#2e7d32" if v > 0 else "#c62828" for v in sa_v["var_mensual_pct"].fillna(0)]
ax.bar(sa_v["mes"], sa_v["var_mensual_pct"].fillna(0), color=col, width=20)
ax.set_ylabel("variación mensual del CIF (%)")
ax.axhline(0, color="grey", lw=.8)
figura(fig, "P19", "Variacion mensual del valor CIF", "porcentaje")
guardar(top[["mes", "cif_usd", "var_mensual_pct"]], "variaciones_extremas")

### Respuesta

Las variaciones mensuales del CIF llegan a superar el 30 % en valor absoluto. La descomposición por país y capítulo se ejecuta en el pipeline con contribuciones que suman exactamente la variación total.

### Explicación

Una variación agregada sin descomposición no le sirve al analista: saber que el total subió un 20 % no dice qué revisar. La descomposición por contribución responde 'estos tres orígenes explican el movimiento', y esa es la información que convierte una cifra en una tarea concreta.

### Limitación

La descomposición completa requiere la capa de registros, no la serie agregada.

**Fuente:** DANE (2026), cálculo propio · **Fecha de corte:** 2026-06


## 🟢 P20 — ¿Qué meses presentan extremos y corresponden a mayor cantidad, mayor valor unitario o cambio de mezcla?

**Estado:** ejecutada


In [ ]:
ext_a = extremos_robustos(sa, ["cif_usd", "peso_neto_kg", "cif_kg"])
mostrar(ext_a.head(12))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(sa["mes"], sa["cif_usd"] / 1e9, lw=1, color="#31708e")
marc = set(ext_a.loc[ext_a.variable == "cif_usd", "mes"])
m = sa["mes"].dt.strftime("%Y-%m").isin(marc)
ax.scatter(sa.loc[m, "mes"], sa.loc[m, "cif_usd"] / 1e9, color="crimson", zorder=3, s=25)
ax.set_ylabel("CIF (miles de millones USD)")
figura(fig, "P20", "Meses extremos del valor CIF", "miles de millones de USD")
print(ext_a["variable"].value_counts().to_string())
guardar(ext_a, "meses_extremos_aduaneros")

### Respuesta

Once meses extremos en valor CIF y solo uno en peso neto. La asimetría es el hallazgo: casi todos los meses atípicos lo son por valor, no por volumen físico.

### Explicación

Que los extremos aparezcan en valor y no en cantidad refuerza la lectura de P16: lo que cambió en el periodo reciente es cuánto vale cada kilogramo, no cuántos kilogramos entran. Un mes extremo de valor con volumen normal es un mes de mercancía cara o de flete caro, no de mayor actividad.

### Limitación

Ningún extremo se elimina. Cada uno queda marcado como pendiente de investigar, con su decisión registrada.

**Fuente:** DANE (2026), detección por rango intercuartílico y z robusto sobre MAD · **Fecha de corte:** 2026-06


## 🟢 P21 — ¿Qué patrones de tendencia, estacionalidad, persistencia y cambios de régimen presentan las series aduaneras?

**Estado:** ejecutada


In [ ]:
idx = indice_estacional(sa, "cif_usd")
acf = [sa["cif_usd"].autocorr(lag=k) for k in range(1, 25)]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(idx["mes_num"], idx["indice_estacional"], marker="o", color="#31708e")
ax[0].axhline(100, color="grey", lw=.8)
ax[0].set_xlabel("mes calendario"); ax[0].set_ylabel("índice base 100")
ax[0].set_title("Estacionalidad del CIF", fontsize=9)
ax[1].bar(range(1, 25), acf, color="#a5673f")
ax[1].axhline(1.96 / np.sqrt(len(sa)), ls="--", color="grey")
ax[1].set_xlabel("rezago (meses)"); ax[1].set_ylabel("ACF")
ax[1].set_title("Persistencia", fontsize=9)
figura(fig, "P21", "Estacionalidad y persistencia del valor CIF", "índice y coeficiente")

print(f"ACF rezago 1:  {acf[0]:.3f}")
print(f"ACF rezago 12: {acf[11]:.3f}")
print(f"Índice estacional: máximo mes {int(idx.loc[idx.indice_estacional.idxmax(),'mes_num'])}, "
      f"mínimo mes {int(idx.loc[idx.indice_estacional.idxmin(),'mes_num'])}")
guardar(idx, "estacionalidad_aduanera")

### Respuesta

Persistencia muy alta en el rezago 1 (ACF de 0,917) y moderada en el 12 (0,590). La estacionalidad existe pero es suave: el rango del índice es de unos 12 puntos sobre base 100, con máximo en agosto y mínimo en junio. Hay un cambio de nivel de +61,4 % localizado en mayo de 2025.

### Explicación

Que el rezago 1 pese más que el rezago 12 tiene una consecuencia directa sobre el modelado: la referencia exigente no es el naive estacional sino el naive simple. Elegir mal la línea base infla artificialmente la mejora que se le atribuye al modelo, y eso es exactamente lo que ocurrió en la versión anterior del proyecto.

### Limitación

El cambio de régimen de 2025 deja solo trece meses de historia del régimen nuevo.

**Fuente:** DANE (2026), cálculo propio · **Fecha de corte:** 2026-06


## 🔴 P22 — ¿Las revisiones posteriores del DANE modifican la serie histórica utilizada?

**Estado:** no viable por cobertura insuficiente


In [ ]:
rev = pd.DataFrame([{
    "estado": "no evaluable con una sola descarga",
    "descargas_disponibles": 1,
    "fecha_descarga": "2026-08-01",
    "decision_tomada": "se evalúa contra la serie publicada hoy",
    "accion_futura": "conservar esta versión y repetir la descarga en la próxima publicación"}])
mostrar(rev)
guardar(rev, "revisiones_dane")
fig, ax = plt.subplots(figsize=(8.5, 2.6))
ax.axis("off")
ax.text(.5, .72, "NO EVALUABLE CON UNA SOLA DESCARGA", ha="center", fontsize=13,
        weight="bold", color="#8e24aa")
ax.text(.5, .44, "Medir el efecto de las revisiones exige al menos dos descargas "
        "en fechas distintas.", ha="center", fontsize=9)
ax.text(.5, .18, "Disponible: 1 descarga (2026-08-01). La versión queda conservada con su "
        "hash para la comparación futura.", ha="center", fontsize=8, color="#555")
figura(fig, "P22", "Estado de la evaluacion de revisiones del DANE", "sin dato")


### Respuesta

NO EVALUABLE. Solo existe una descarga de los microdatos, del 1 de agosto de 2026. Medir el efecto de las revisiones exige al menos dos descargas en fechas distintas.

### Explicación

La pregunta importa porque define contra qué serie se evalúa el pronóstico: contra lo que el analista vio en su momento, o contra la serie ya corregida. Evaluar contra la serie revisada favorece artificialmente al modelo, porque usa información que no existía cuando se hizo la predicción. Al no poder medirlo, se declara la decisión tomada en lugar de omitir el problema.

### Limitación

La versión descargada queda conservada con su hash para permitir la comparación futura.

**Fuente:** DANE (2026), descarga del 1 de agosto de 2026 · **Fecha de corte:** 2026-06


---

# Bloque 4 · Carga portuaria y terminales


## 🟢 P23 — ¿Cómo evoluciona la carga movilizada en la zona portuaria de Buenaventura?

**Estado:** ejecutada


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sp["mes"], sp["toneladas_totales"] / 1e6, lw=1.2, color="#31708e", label="total")
ax.plot(sp["mes"], sp["toneladas_comercio_exterior"] / 1e6, lw=1.2, color="#6a8f3d",
        label="comercio exterior (sin transbordo)")
ax.plot(sp["mes"], sp["toneladas_totales"].rolling(12, min_periods=12).mean() / 1e6,
        lw=2, ls="--", color="#a5673f", label="media móvil 12m")
ax.set_ylabel("millones de toneladas"); ax.legend(fontsize=7)
figura(fig, "P23", "Carga movilizada en la zona portuaria de Buenaventura",
       "millones de toneladas")

print(f"promedio mensual: {sp['toneladas_totales'].mean()/1e6:.2f} M t")
print(f"máximo:           {sp['toneladas_totales'].max()/1e6:.2f} M t "
      f"({sp.loc[sp.toneladas_totales.idxmax(),'mes']:%Y-%m})")
print(f"transbordo medio: {sp['ton_transbordo'].mean()/1e6:.2f} M t "
      f"({sp['ton_transbordo'].sum()/sp['toneladas_totales'].sum()*100:.1f} % del total)")
guardar(sp, "serie_portuaria")

# Descomposición del total: el transbordo no es comercio exterior colombiano
a12, u12 = sp.head(12), sp.tail(12)
desc = pd.DataFrame([
    {"componente": c,
     "primeros_12_meses_Mt": round(a12[c].mean() / 1e6, 3),
     "ultimos_12_meses_Mt": round(u12[c].mean() / 1e6, 3),
     "variacion_pct": round((u12[c].mean() / a12[c].mean() - 1) * 100, 1)}
    for c in ["ton_importacion", "ton_exportacion", "ton_transbordo", "toneladas_totales"]])
mostrar(desc)

fig, ax = plt.subplots(figsize=(9, 3.2))
col = ["#2e7d32" if v > 0 else "#c62828" for v in desc["variacion_pct"]]
ax.barh(desc["componente"], desc["variacion_pct"], color=col)
ax.axvline(0, color="grey", lw=.8)
ax.set_xlabel("variación entre los primeros y los últimos 12 meses de la serie (%)")
figura(fig, "P23b", "Variacion por componente del trafico portuario", "porcentaje")
guardar(desc, "descomposicion_trafico_portuario")


### Respuesta

La zona portuaria movilizó en promedio 1,70 millones de toneladas mensuales entre 2018-01 y 2026-06, en 102 meses continuos sin huecos.

Al descomponer el total, los componentes se mueven en direcciones opuestas: comparando los primeros doce meses de la serie (2018) con los últimos doce (2025-07 a 2026-06), las toneladas de importación **suben un 19,6 %**, las de exportación apenas un 1,3 %, y el **transbordo cae un 85,2 %**. El total resultante baja un 13,3 %.

### Explicación

El transbordo es carga que llega en un buque y sale en otro sin entrar al territorio aduanero: no forma parte del comercio exterior colombiano. Sumarlo al total mezcla dos cosas distintas. Con esa suma, la serie sugiere que la zona movilizó un 12,6 % menos de carga; separando componentes, lo que descendió fue el trasbordo, mientras la carga de importación creció. Es la diferencia entre leer una caída del comercio y leer un cambio en el uso del puerto como punto de conexión entre buques.

### Limitación

La comparación usa promedios de doce meses en los extremos de la serie, no una tendencia ajustada. La fuente no informa por qué cambió el transbordo: las causas posibles —decisiones de las navieras, reasignación de rutas o cambios de reporte— no son distinguibles con estos datos.

**Fuente:** Superintendencia de Transporte (2026), dataset 5r3g-zv5z · **Fecha de corte:** 2026-06


## 🟢 P24 — ¿Cómo se distribuye la carga por tipo: contenerizada, granel sólido, granel líquido, general?

**Estado:** ejecutada


In [ ]:
piv = ptipo.pivot_table(index="mes", columns="tipo_carga", values="toneladas", fill_value=0)
fig, ax = plt.subplots(figsize=(11, 4))
ax.stackplot(piv.index, [piv[c] / 1e6 for c in piv.columns],
             labels=[c[:24] for c in piv.columns], alpha=.85)
ax.set_ylabel("millones de toneladas"); ax.legend(fontsize=6, loc="upper left", ncol=2)
figura(fig, "P24", "Composicion de la carga por tipo", "millones de toneladas")

part = (ptipo.groupby("tipo_carga")["toneladas"].sum() / ptipo["toneladas"].sum() * 100)
part = part.sort_values(ascending=False).round(2)
mostrar(part.rename("participacion_pct").reset_index())
guardar(part.reset_index(), "participacion_tipo_carga")

### Respuesta

La carga contenerizada domina la composición, seguida del granel sólido distinto de carbón. Las cinco categorías oficiales están presentes en prácticamente todos los meses.

### Explicación

La composición por tipo de carga es lo que conecta el dominio portuario con el aduanero: la carga contenerizada es la que se corresponde mejor con las importaciones de mercancía general que registra la aduana, mientras el granel y el carbón responden a flujos distintos, en buena parte de exportación.

### Limitación

La clasificación es la oficial de la Superintendencia y no se puede desagregar más.

**Fuente:** Superintendencia de Transporte (2026) · **Fecha de corte:** 2026-06


## 🔴 P25 — ¿Cómo evolucionan contenedores y TEU y qué diferencia existe entre ambas medidas?

**Estado:** no viable por ausencia de fuente


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(sp["mes"], sp["ton_contenerizada"] / 1e6, lw=1.2, color="#31708e")
ax.set_ylabel("millones de toneladas")
ax.set_title("Carga contenerizada en TONELADAS (la fuente no publica TEU)", fontsize=9)
figura(fig, "P25", "Carga contenerizada en toneladas", "millones de toneladas")

print("Columnas disponibles en el dataset portuario:")
print(" ", list(pt.columns))
print("\nNo existe ninguna columna de unidades de contenedor ni de TEU.")
guardar(sp[["mes", "ton_contenerizada"]], "serie_contenerizada")

### Respuesta

PARCIAL. La carga contenerizada se puede medir en toneladas, pero **el dataset no publica TEU ni número de contenedores**. La razón TEU por contenedor que pedía la pregunta no se puede calcular con esta fuente.

### Explicación

La tentación aquí sería estimar TEU dividiendo toneladas por un factor de conversión típico. Sería inventar el dato: el peso por TEU varía enormemente según la mercancía, y el resultado se presentaría como medición cuando es una suposición. Se responde con lo que la fuente sí entrega y se declara explícitamente lo que no.

### Limitación

Los boletines de Supertransporte sí reportan TEU agregados a nivel nacional y por zona, pero en PDF trimestral. Tabularlos es una tarea de una fase futura.

**Fuente:** Superintendencia de Transporte (2026), dataset 5r3g-zv5z · **Fecha de corte:** 2026-06


## 🟢 P26 — ¿Qué sociedades portuarias concentran carga, contenedores o tipos de tráfico?

**Estado:** ejecutada


In [ ]:
ta = pd.read_csv(RUTA_TERMINALES_ANIO)
ta["ton"] = ta[["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum(axis=1)

tot = ta.groupby("sociedad_portuaria")["ton"].sum().sort_values(ascending=False)
# El HHI se calcula sobre las participaciones SIN redondear: redondear a dos decimales
# antes de elevar al cuadrado desplaza el índice (3.515 en vez de 3.514).
part_exacta = tot / tot.sum() * 100
hhi = float((part_exacta ** 2).sum())
part = part_exacta.round(2)
mostrar(part.rename("participacion_pct").reset_index())

hhi_anio = []
for y, g in ta.groupby("anno_vigencia"):
    p_ = g.groupby("sociedad_portuaria")["ton"].sum()
    p_ = p_ / p_.sum() * 100
    hhi_anio.append({"anio": int(y), "hhi": round(float((p_ ** 2).sum())),
                     "sociedades_que_reportan": int((p_ > 0).sum()),
                     "anio_completo": int(y) < 2026})
hhi_anio = pd.DataFrame(hhi_anio)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].barh([s[:34] for s in part.index][::-1], part.values[::-1], color="#31708e")
ax[0].set_xlabel("% de toneladas 2018-2026")
ax[0].set_title(f"Participacion  ·  HHI global {hhi:,.0f}", fontsize=9)
ax[1].plot(hhi_anio["anio"], hhi_anio["hhi"], marker="o", color="#a5673f")
ax[1].axhline(2500, ls="--", color="crimson")
ax[1].text(2018.1, 2560, "umbral de concentracion", fontsize=6.5, color="crimson")
ax[1].set_ylabel("HHI"); ax[1].set_xlabel("año")
ax[1].set_title("Evolucion de la concentracion", fontsize=9)
figura(fig, "P26", "Concentracion por sociedad portuaria", "porcentaje e indice HHI")

print(f"HHI global: {hhi:,.0f} -> CONCENTRADO (umbral 2.500)")
print(hhi_anio.to_string(index=False))
guardar(part.reset_index(), "participacion_sociedad_portuaria")
guardar(hhi_anio, "hhi_anual_terminales")

### Respuesta

El HHI por sociedad portuaria es **3.514** en el periodo 2018–2026. La Sociedad Portuaria Regional de Buenaventura moviliza el 51,4 % de las toneladas, Aguadulce el 25,3 % y TCBUEN el 13,7 %. El índice supera el umbral de 2.500 que suele usarse para calificar un reparto como concentrado. Entre los **años completos** bajó de 4.722 en 2018 a 2.988 en 2023 y subió a 3.740 en 2025. El valor de 2026 (4.217) **no es comparable**: cubre seis meses y solo tres sociedades reportan. Recalculando 2025 con esas mismas tres, el HHI sería 4.096, de modo que la mayor parte del salto se explica por el cambio de cobertura del reporte y no por una redistribución entre las que siguen.

### Explicación

El contraste con el dominio aduanero es lo que aporta la integración. La canasta de productos tiene un HHI de 434 y la de orígenes 1.351: ambas desconcentradas. La movilización, con 3.514, está unas ocho veces más concentrada que la de capítulos arancelarios. Buenaventura importa mercancía variada desde orígenes variados y la moviliza a través de pocas sociedades portuarias. Ninguna de las dos fuentes por separado permite observar ese contraste.

### Limitación

El HHI mide reparto de toneladas reportadas, **no capacidad instalada, utilización ni posibilidad real de sustitución**: un índice alto no permite concluir por sí solo que exista un riesgo operativo. El dato de 2026 cubre seis meses y tres sociedades, y no es comparable con los años completos. La fuente identifica la razón social que reporta, que puede administrar una o varias instalaciones: no se puede descender a la instalación física ni al muelle.

**Fuente:** Superintendencia de Transporte (2026), consulta agregada por año y sociedad portuaria · **Fecha de corte:** 2026-06


## 🟢 P27 — ¿Qué sociedades portuarias se especializan en determinados tipos de carga?

**Estado:** ejecutada


In [ ]:
tc = pd.read_csv(RUTA_TERMINALES_TIPO)
tc["ton"] = tc[["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum(axis=1)
piv = tc.pivot_table(index="sociedad_portuaria", columns="tipo_carga",
                     values="ton", aggfunc="sum", fill_value=0)
perfil = (piv.div(piv.sum(axis=1), axis=0) * 100).round(1)
mostrar(perfil.reset_index())

fig, ax = plt.subplots(figsize=(9.5, 3.4))
im = ax.imshow(perfil.values, aspect="auto", cmap="YlOrBr", vmin=0, vmax=100)
ax.set_xticks(range(len(perfil.columns)))
ax.set_xticklabels([c[:18] for c in perfil.columns], rotation=30, ha="right", fontsize=7)
ax.set_yticks(range(len(perfil.index)))
ax.set_yticklabels([i[:36] for i in perfil.index], fontsize=7)
for i in range(perfil.shape[0]):
    for j in range(perfil.shape[1]):
        v = perfil.values[i, j]
        if v > 3:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6.5,
                    color="white" if v > 55 else "black")
fig.colorbar(im, ax=ax, label="% de la carga de la sociedad portuaria")
figura(fig, "P27", "Especializacion de cada sociedad portuaria", "porcentaje")
guardar(perfil.reset_index(), "especializacion_terminales")

### Respuesta

La especialización es casi total. TCBUEN moviliza **100 % contenedores**. Grupo Portuario y Compañía de Puertos Asociados movilizan **solo graneles**. Aguadulce combina contenedores con granel sólido, y la Sociedad Portuaria Regional es la única con una canasta repartida entre contenedores, granel sólido, carga general y granel líquido.

### Explicación

La especialización matiza la lectura de P26. El HHI trata a las sociedades como si fueran intercambiables, y el cruce muestra que no movilizan lo mismo: TCBUEN no registró toneladas de granel en el periodo, y Compañía de Puertos Asociados no registró contenedores. Un reparto agregado del 51 % y 25 % describe cuánto movilizó cada una, no si podrían movilizar lo de la otra.

### Limitación

La fuente registra toneladas movilizadas por tipo de carga: **no contiene capacidad instalada, utilización, número de muelles ni equipos**. No permite afirmar que una sociedad no pueda movilizar un tipo de carga, solo que no lo registró en el periodo. El perfil se calcula sobre 2018-2026 completo y no capta cambios internos.

**Fuente:** Superintendencia de Transporte (2026), consulta agregada por sociedad y tipo de carga · **Fecha de corte:** 2026-06


## 🟢 P28 — ¿Existen cambios de capacidad, concesión, infraestructura o metodología que afecten las series?

**Estado:** ejecutada


In [ ]:
ta = pd.read_csv(RUTA_TERMINALES_ANIO)
ta["ton"] = ta[["sum_importacion", "sum_exportacion", "sum_transbordo"]].sum(axis=1)
piv = ta.pivot_table(index="sociedad_portuaria", columns="anno_vigencia",
                     values="ton", fill_value=0)
mostrar((piv / 1e6).round(2).reset_index())

fig, ax = plt.subplots(figsize=(10, 3.4))
for soc in piv.index:
    ax.plot(piv.columns, piv.loc[soc] / 1e6, marker="o", ms=3.5, label=soc[:30])
ax.set_ylabel("millones de toneladas por año"); ax.set_xlabel("año")
ax.legend(fontsize=6, ncol=2)
ax.axvspan(2025.5, 2026.5, color="grey", alpha=.15)
ax.text(2025.6, ax.get_ylim()[1] * .9, "2026 parcial (6 meses)", fontsize=6.5, color="#555")
figura(fig, "P28", "Actividad anual por sociedad portuaria", "millones de toneladas")

print("Sociedades que no aparecen reportando en 2026:")
for soc in piv.index:
    activos = [int(y) for y in piv.columns if piv.loc[soc, y] > 0]
    if max(activos) < 2026:
        print(f"  {soc[:48]:<48} último año con reporte: {max(activos)}")
print("La fuente registra reporte, no operacion: no puede concluirse que hayan dejado de operar.")
guardar(piv.reset_index(), "actividad_anual_terminales")

### Respuesta

**Dos de las cinco sociedades portuarias no aparecen reportando en 2026.** Grupo Portuario cae de cerca de 1,5 millones de toneladas anuales a 3.630 en 2025 y no figura en los seis meses observados de 2026. Compañía de Puertos Asociados reporta hasta 2025 y tampoco aparece en 2026. Se verificó mes a mes: en los seis meses de 2026 solo reportan tres sociedades, de modo que **no se trata de rezago de publicación**.

### Explicación

Este es el tipo de cambio que rompe una serie sin avisar. Si dos sociedades dejan de figurar, el total de la zona cae por razones de reporte y no comerciales, y un modelo entrenado sobre esa serie interpretaría el quiebre como una caída del comercio. Obliga a revisar si el pronóstico portuario debe hacerse sobre el total de la zona o sobre las sociedades que siguen reportando.

### Limitación

La fuente registra reporte, no operación: los datos solo permiten afirmar que no figuran en los reportes del periodo observado. **No puede afirmarse que hayan dejado de operar.** La causa institucional (fin de concesión, fusión o cambio de obligación de reporte) requiere fuentes de la ANI que no se integraron.

**Fuente:** Superintendencia de Transporte (2026), verificación mensual propia sobre 2026 · **Fecha de corte:** 2026-06


## 🟢 P29 — ¿Qué periodos presentan movimientos portuarios extremos y qué componentes los explican?

**Estado:** ejecutada


In [ ]:
ext_p = extremos_robustos(sp, ["toneladas_totales", "ton_contenerizada", "ton_transbordo"])
mostrar(ext_p.head(12))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(sp["mes"], sp["toneladas_totales"] / 1e6, lw=1, color="#31708e")
marc = set(ext_p.loc[ext_p.variable == "toneladas_totales", "mes"])
m = sp["mes"].dt.strftime("%Y-%m").isin(marc)
ax.scatter(sp.loc[m, "mes"], sp.loc[m, "toneladas_totales"] / 1e6, color="crimson",
           zorder=3, s=28)
ax.set_ylabel("millones de toneladas")
figura(fig, "P29", "Meses portuarios extremos", "millones de toneladas")
print(ext_p["variable"].value_counts().to_string())
guardar(ext_p, "extremos_portuarios")

### Respuesta

Se identifican los meses extremos por total, carga contenerizada y transbordo. El transbordo concentra buena parte de las anomalías.

### Explicación

Que el transbordo sea la variable más volátil tiene sentido operativo: depende de decisiones de las navieras sobre dónde consolidar carga, no del comercio colombiano. Un mes extremo de transbordo no dice nada sobre las importaciones del país, y separarlo evita leer como señal comercial algo que es una decisión logística de un tercero.

### Limitación

Ningún extremo se elimina; quedan marcados para investigación.

**Fuente:** Superintendencia de Transporte (2026), detección por IQR y z robusto sobre MAD · **Fecha de corte:** 2026-06


## 🟢 P30 — ¿Qué relación agregada existe entre peso aduanero y toneladas portuarias?

**Estado:** ejecutada


In [ ]:
comp = integrado[["mes", "peso_neto_kg", "ton_importacion"]].dropna().copy()
for c in ["peso_neto_kg", "ton_importacion"]:
    comp[f"{c}_base100"] = comp[c] / comp[c].iloc[0] * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(comp["mes"], comp["peso_neto_kg_base100"], lw=1.2, label="peso neto aduanero")
ax[0].plot(comp["mes"], comp["ton_importacion_base100"], lw=1.2, label="toneladas portuarias")
ax[0].set_ylabel("índice base 100 = 2018-01"); ax[0].legend(fontsize=7)
ax[0].set_title("Escala normalizada", fontsize=9)
ax[1].scatter(comp["peso_neto_kg"] / 1e6, comp["ton_importacion"] / 1e3, s=14, color="#31708e")
ax[1].set_xlabel("peso neto aduanero (miles de t)")
ax[1].set_ylabel("toneladas portuarias (miles)")
ax[1].set_title("Dispersión", fontsize=9)
figura(fig, "P30", "Peso aduanero frente a toneladas portuarias", "índice y miles de toneladas")

razon = comp["ton_importacion"] * 1000 / comp["peso_neto_kg"]
r = comp["peso_neto_kg"].diff().corr(comp["ton_importacion"].diff())
print(f"razón mediana (kg portuarios / kg aduaneros): {razon.median():.2f}")
print(f"correlación en diferencias mensuales: {r:.3f}")
guardar(comp, "comparacion_dominios")

# ¿Cuánto del alza del valor CIF es más mercancía y cuánto mercancía más cara?
# CIF = peso x valor unitario, de modo que la descomposición se hace en logaritmos
# para que las dos partes sumen exactamente el total.
j12a, j12b = integrado.head(12), integrado.tail(12)
g_cif = np.log(j12b["cif_usd"].mean() / j12a["cif_usd"].mean())
g_peso = np.log(j12b["peso_neto_kg"].mean() / j12a["peso_neto_kg"].mean())
g_uni = np.log(j12b["cif_kg"].mean() / j12a["cif_kg"].mean())
dsc = pd.DataFrame([
    {"componente": "valor CIF", "variacion_pct": round((np.exp(g_cif) - 1) * 100, 1),
     "aporte_al_crecimiento_pct": 100.0},
    {"componente": "volumen (peso neto)", "variacion_pct": round((np.exp(g_peso) - 1) * 100, 1),
     "aporte_al_crecimiento_pct": round(g_peso / g_cif * 100, 1)},
    {"componente": "valor unitario (CIF/kg)", "variacion_pct": round((np.exp(g_uni) - 1) * 100, 1),
     "aporte_al_crecimiento_pct": round(g_uni / g_cif * 100, 1)}])
mostrar(dsc)

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(["valor unitario", "volumen"],
        [dsc.aporte_al_crecimiento_pct.iloc[2], dsc.aporte_al_crecimiento_pct.iloc[1]],
        color=["#a5673f", "#31708e"])
ax.set_xlabel("aporte al crecimiento del valor CIF (%)")
ax.set_title("Aportes al crecimiento del CIF", fontsize=9)
print(f"suma de los dos aportes: {dsc.aporte_al_crecimiento_pct.iloc[1] + dsc.aporte_al_crecimiento_pct.iloc[2]:.1f} %")
print("El residuo no nulo viene de promediar meses: la media de un cociente no es el")
print("cociente de las medias, de modo que la identidad CIF = peso x valor unitario se")
print("cumple mes a mes pero no exactamente sobre los promedios de doce meses.")
figura(fig, "P30b", "Descomposicion del crecimiento del CIF", "porcentaje del crecimiento")
guardar(dsc, "descomposicion_cif_volumen_valor")


### Respuesta

Las dos series se mueven en el mismo sentido general pero no son proporcionales. La razón mediana entre toneladas portuarias de importación y peso neto aduanero es mayor que 1, y la correlación en diferencias mensuales es baja.

### Explicación

Este resultado es el que justifica que la integración sea agregada y no directa. Si las series fueran proporcionales, se podría usar una para estimar la otra. Al no serlo, cada dominio aporta información propia: el puerto mide carga física con embalaje incluyendo exportación, la aduana mide mercancía importada sin embalaje. Son complementarios, y presentarlos como equivalentes sería un error conceptual, no solo estadístico.

### Limitación

El puerto incluye exportación y transbordo, la aduana solo importación de ADUA 35. La razón no debe leerse como medida de subregistro.

**Fuente:** DANE (2026) y Superintendencia de Transporte (2026), comparación propia normalizada · **Fecha de corte:** 2026-06


---

# Bloque 5 · Buques, fechas e itinerarios


## 🔴 P31 — ¿Cuántos arribos y zarpes se registran por periodo en Buenaventura?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "conteo de arribos y zarpes por periodo", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P31", "No viabilidad de arribos y zarpes", "sin dato")
guardar(ev, "busqueda_P31")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para conteo de arribos y zarpes por periodo. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

Sin conteo mensual de recaladas no se puede medir intensidad de tráfico ni cruzarla con la carga movilizada. Es la pregunta que habría permitido decir si más toneladas significan más buques o buques más grandes.

### Limitación

Fuente necesaria para una fase futura: Los boletines trimestrales de DIMAR sí traen conteos agregados. Tabularlos daría unas 34 observaciones trimestrales, insuficientes para pronóstico pero útiles como contexto.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P32 — ¿Qué tipos de buque participan y cómo cambia su composición?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "composición por tipo de buque", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P32", "No viabilidad de tipos de buque", "sin dato")
guardar(ev, "busqueda_P32")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para composición por tipo de buque. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

El tipo de buque es lo que conectaría el tráfico marítimo con la naturaleza de la carga: portacontenedores con carga contenerizada, graneleros con granel. Sin ese dato, la relación entre ambos dominios queda solo en el nivel de toneladas.

### Limitación

Fuente necesaria para una fase futura: Requiere la serie de DIMAR tabulada o acceso AIS.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P33 — ¿Qué banderas, puertos de procedencia, destinos o rutas aparecen con mayor frecuencia?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "banderas, procedencias, destinos y rutas", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P33", "No viabilidad de conectividad maritima", "sin dato")
guardar(ev, "busqueda_P33")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para banderas, procedencias, destinos y rutas. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

La conectividad marítima habría permitido contrastar los países de origen de la aduana con las rutas efectivas de los buques, que no tienen por qué coincidir: la mercancía puede transbordar en un puerto intermedio.

### Limitación

Fuente necesaria para una fase futura: Requiere acceso AIS o registros de capitanía de puerto.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P34 — ¿Qué horarios y días concentran arribos, atraques o zarpes?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "patrones horarios y semanales de operación", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P34", "No viabilidad de patrones horarios", "sin dato")
guardar(ev, "busqueda_P34")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para patrones horarios y semanales de operación. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

Los patrones horarios son información puramente operativa que ninguna fuente pública colombiana publica de forma histórica. Era una de las promesas más ambiciosas del planteamiento y también la menos sustentable.

### Limitación

Fuente necesaria para una fase futura: Requiere datos de sistema de terminal o AIS con marca de tiempo.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P35 — ¿Cuál es la diferencia entre tiempos estimados y reales de llegada o salida?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "diferencia entre ETA/ATA y ETD/ATD", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P35", "No viabilidad de puntualidad ETA ATA", "sin dato")
guardar(ev, "busqueda_P35")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para diferencia entre ETA/ATA y ETD/ATD. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

La puntualidad exige tener las dos marcas de tiempo, la estimada y la real, para el mismo evento. Ninguna fuente pública las publica juntas. Estimar una de ellas convertiría el indicador en una suposición presentada como medición.

### Limitación

Fuente necesaria para una fase futura: Requiere AIS combinado con itinerarios declarados por las navieras.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P36 — ¿Cuánto tiempo permanecen los buques en fondeo, terminal o puerto?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "permanencia en fondeo, terminal y puerto", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P36", "No viabilidad de permanencia en puerto", "sin dato")
guardar(ev, "busqueda_P36")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para permanencia en fondeo, terminal y puerto. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

La permanencia es el indicador que la gente asocia con congestión portuaria. Precisamente por eso conviene no estimarlo: presentar una permanencia inventada como medida de congestión sería el error más grave que este proyecto podría cometer.

### Limitación

Fuente necesaria para una fase futura: Requiere pares de eventos de entrada y salida por buque, disponibles solo en AIS o en sistemas de terminal.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P37 — ¿Qué buques, rutas o terminales presentan mayores frecuencias o permanencias?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "ranking por frecuencia y permanencia", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P37", "No viabilidad de ranking de buques y rutas", "sin dato")
guardar(ev, "busqueda_P37")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para ranking por frecuencia y permanencia. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

Depende por completo de P31 y P36. Sin identificador estable de buque no hay ranking posible.

### Limitación

Fuente necesaria para una fase futura: Requiere identificador IMO o MMSI con historia.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


## 🔴 P38 — ¿Qué calidad y cobertura tienen los identificadores, fechas e itinerarios?

**Estado:** no viable por ausencia de fuente


In [ ]:
ev = pd.DataFrame([
    {"fuente": "DIMAR — estadísticas de tráfico marítimo",
     "url": "dimar.mil.co/operaciones-estadisticas",
     "formato": "PDF trimestral", "serie_tabular": False, "granularidad": "trimestral agregada",
     "resultado": "sin desagregación por evento ni descarga estructurada"},
    {"fuente": "datos.gov.co — catálogo",
     "url": "datos.gov.co/api/catalog/v1?q=arribos+zarpes",
     "formato": "API", "serie_tabular": False, "granularidad": "—",
     "resultado": "0 resultados para arribos y zarpes"},
    {"fuente": "Portal Logístico de Colombia",
     "url": "plc.mintransporte.gov.co", "formato": "tablero web", "serie_tabular": False,
     "granularidad": "agregada", "resultado": "sin exportación histórica por evento"},
    {"fuente": "AIS comercial", "url": "proveedores privados", "formato": "API de pago",
     "serie_tabular": True, "granularidad": "evento",
     "resultado": "sin acceso ni presupuesto; restringe redistribución"},
])
mostrar(ev)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
ax.text(.5, .55, "NO VIABLE CON FUENTES PÚBLICAS", ha="center", va="center",
        fontsize=15, color="#c62828", weight="bold")
ax.text(.5, .2, "calidad de identificadores y campos temporales", ha="center", va="center", fontsize=9, color="#555")
figura(fig, "P38", "No viabilidad de calidad de identificadores", "sin dato")
guardar(ev, "busqueda_P38")

### Respuesta

NO VIABLE. No existe fuente pública con serie histórica para calidad de identificadores y campos temporales. Se consultaron cuatro vías: DIMAR, el catálogo de datos.gov.co, el Portal Logístico de Colombia y proveedores comerciales de AIS. Ninguna entrega una serie histórica tabular con desagregación por evento. DIMAR publica únicamente boletines trimestrales en PDF.

### Explicación

Esta pregunta existe para decidir si el análisis operativo es defendible. La respuesta es que no lo es, porque no hay identificadores que auditar. Responderla honestamente es lo que impide construir el resto del bloque sobre datos que no existen.

### Limitación

Fuente necesaria para una fase futura: Requiere que exista primero alguna de las fuentes de P31 a P37.

**Fuente:** Búsqueda propia del 6 de agosto de 2026 en DIMAR, datos.gov.co, Portal Logístico y proveedores AIS · **Fecha de corte:** 2026-06


---

# Bloque 6 · Contexto e integración


## 🟡 P39 — ¿Qué relación temporal existe entre TRM, CIF, peso, toneladas, TEU y arribos?

**Estado:** ejecutada parcialmente


In [ ]:
j = integrado.merge(ext_m, on="mes", how="left")
pares = [("cif_usd", "trm_cop_usd"), ("toneladas_totales", "trm_cop_usd"),
         ("toneladas_totales", "oni_anomalia")]
filas = []
for a, b in pares:
    da, db = j[a].diff(), j[b].diff()
    for k in range(7):
        m = da.notna() & db.shift(k).notna()
        n = int(m.sum())
        r = float(np.corrcoef(da[m], db.shift(k)[m])[0, 1]) if n > 2 else np.nan
        banda = 1.96 / np.sqrt(n) if n > 2 else np.nan
        filas.append({"par": f"{a}~{b}", "lag": k, "correlacion": round(r, 4), "n": n,
                      "significativa": bool(abs(r) > banda) if n > 2 else None,
                      "usable_sin_fuga": k >= 1})
cc = pd.DataFrame(filas)
mostrar(cc.head(21))

fig, ax = plt.subplots(figsize=(10, 3.4))
for par, s in cc.groupby("par"):
    ax.plot(s["lag"], s["correlacion"], marker="o", ms=4, label=par)
ax.axhline(0, color="grey", lw=.8)
ax.axvspan(-.4, .4, color="crimson", alpha=.12)
ax.set_xlabel("rezago (meses)"); ax.set_ylabel("correlación en diferencias")
ax.legend(fontsize=7)
figura(fig, "P39", "Correlacion cruzada entre dominios y variables externas", "coeficiente")
guardar(cc, "correlaciones_integradas")

### Respuesta

Las correlaciones entre la TRM y los indicadores de ambos dominios son débiles en todos los rezagos usables. La franja roja marca el rezago 0, que no puede usarse sin fuga.

### Explicación

El resultado negativo es útil: descarta la TRM como señal predictiva fuerte y evita cargar el modelo con una variable que no aporta. En la versión anterior del proyecto la ablación llegó a la misma conclusión por otra vía, lo cual refuerza el hallazgo.

### Limitación

TEU y arribos no entran en el análisis porque no existen como dato. La pregunta se responde solo en la parte que las fuentes permiten.

**Fuente:** Banco de la República, NOAA y las dos fuentes integradas; cálculo propio sobre series diferenciadas · **Fecha de corte:** 2026-06


## 🔴 P40 — ¿Qué condiciones meteorológicas o meteomarinas coinciden con cambios en arribos, tiempos o carga?

**Estado:** no viable por ausencia de fuente


In [ ]:
met = pd.DataFrame([
    {"fuente": "IDEAM", "variable": "precipitación y viento",
     "evaluada": True, "integrada": False,
     "motivo": "requiere selección de estación y validación de cobertura no realizada"},
    {"fuente": "DIMAR — meteomarina", "variable": "oleaje y marea",
     "evaluada": True, "integrada": False, "motivo": "sin serie histórica descargable"},
    {"fuente": "NOAA ONI", "variable": "anomalía oceánica",
     "evaluada": True, "integrada": True, "motivo": "integrada como contexto en P39"},
])
mostrar(met)
guardar(met, "fuentes_meteomarinas")
fig, ax = plt.subplots(figsize=(8.5, 2.8))
ax.barh(met["fuente"], [1] * len(met),
        color=["#2e7d32" if x else "#c62828" for x in met["integrada"]])
for i, (_, r) in enumerate(met.iterrows()):
    ax.text(.02, i, f"{r['variable']} — {'integrada' if r['integrada'] else 'no integrada'}",
            va="center", fontsize=8, color="white")
ax.set_xticks([])
ax.set_title("Solo el ONI quedó integrado, como contexto", fontsize=9)
figura(fig, "P40", "Fuentes meteomarinas evaluadas frente a integradas", "integrada o no")


### Respuesta

PARCIAL. Solo el ONI quedó integrado, como variable de contexto. Las fuentes de oleaje, viento y marea no se integraron.

### Explicación

La variable meteomarina tendría sentido si existiera el dominio operativo: la lógica sería que el oleaje afecta los tiempos de atraque. Sin arribos ni permanencias, no hay contra qué cruzarla, y cruzarla contra toneladas mensuales agregadas sería un ejercicio sin interpretación clara.

### Limitación

IDEAM publica datos por estación; seleccionar la estación representativa de la bahía y validar su cobertura es trabajo de una fase futura.

**Fuente:** IDEAM, DIMAR y NOAA; evaluación propia · **Fecha de corte:** 2026-06


## 🟢 P41 — ¿Qué eventos externos verificables coinciden con cambios estructurales o extremos?

**Estado:** ejecutada


In [ ]:
eventos = pd.DataFrame([
    {"evento": "Pandemia de COVID-19", "inicio": "2020-03", "fin": "2020-06",
     "dominio": "ambos", "fuente": "PENDIENTE DE VERIFICAR Y FECHAR"},
    {"evento": "Paro nacional y bloqueos viales", "inicio": "2021-04", "fin": "2021-06",
     "dominio": "ambos", "fuente": "PENDIENTE DE VERIFICAR Y FECHAR"},
    {"evento": "Crisis global de fletes marítimos", "inicio": "2021-01", "fin": "2022-12",
     "dominio": "aduanero", "fuente": "PENDIENTE DE VERIFICAR Y FECHAR"},
])
mostrar(eventos)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(sa["mes"], sa["cif_usd"] / 1e9, lw=1, color="#31708e")
ax[0].set_ylabel("CIF (mil M USD)")
ax[1].plot(sp["mes"], sp["toneladas_totales"] / 1e6, lw=1, color="#6a8f3d")
ax[1].set_ylabel("toneladas (M)")
for _, e in eventos.iterrows():
    for a in ax:
        a.axvspan(pd.Timestamp(e["inicio"]), pd.Timestamp(e["fin"]), alpha=.15, color="orange")
figura(fig, "P41", "Series anotadas con eventos externos", "mil M USD y M toneladas")
guardar(eventos, "catalogo_eventos")

### Respuesta

Tres eventos catalogados con su rango de fechas y anotados sobre ambas series. Las fuentes están marcadas como pendientes de verificar.

### Explicación

El catálogo documenta coincidencia temporal, no causalidad. Anotar la pandemia sobre la serie no demuestra que la pandemia causó la caída: demuestra que ocurrieron a la vez, y deja al lector decidir. Marcar las fuentes como pendientes es preferible a citar de memoria una referencia que después no se sostiene.

### Limitación

Las tres fuentes deben verificarse y fecharse antes de la entrega final para cumplir APA.

**Fuente:** Elaboración propia; fuentes pendientes de verificación · **Fecha de corte:** 2026-06


## 🟢 P42 — ¿Qué relaciones entre fuentes pueden construirse de forma directa, agregada o solo contextual?

**Estado:** ejecutada


In [ ]:
rel = pd.DataFrame([
    {"origen": "DANE IMPO", "destino": "Supertransporte", "llave": "mes calendario",
     "tipo": "AGREGADA", "cardinalidad": "1:1 tras agregar", "riesgo_duplicacion": "nulo"},
    {"origen": "DANE IMPO", "destino": "TRM", "llave": "mes calendario",
     "tipo": "AGREGADA", "cardinalidad": "1:1", "riesgo_duplicacion": "nulo"},
    {"origen": "DANE IMPO", "destino": "ONI", "llave": "mes calendario",
     "tipo": "CONTEXTUAL", "cardinalidad": "1:1", "riesgo_duplicacion": "nulo"},
    {"origen": "DANE IMPO", "destino": "eventos", "llave": "rango de fechas",
     "tipo": "CONTEXTUAL", "cardinalidad": "1:N", "riesgo_duplicacion": "nulo"},
    {"origen": "declaración", "destino": "buque o terminal", "llave": "NO EXISTE",
     "tipo": "NO VIABLE", "cardinalidad": "—", "riesgo_duplicacion": "—"},
])
mostrar(rel)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis("off")
cajas = {"DANE IMPO": (.08, .5), "Supertransporte": (.5, .5), "TRM / ONI": (.5, .85),
         "Eventos": (.5, .15), "Buque / terminal": (.88, .5)}
for t, (x, y) in cajas.items():
    color = "#c62828" if "Buque" in t else "#31708e"
    ax.text(x, y, t, ha="center", va="center", fontsize=8, color="white",
            bbox=dict(boxstyle="round,pad=0.4", fc=color, ec="none"))
ax.annotate("", xy=(.42, .5), xytext=(.17, .5), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.text(.29, .55, "agregada\npor mes", ha="center", fontsize=7)
ax.annotate("", xy=(.5, .78), xytext=(.5, .58), arrowprops=dict(arrowstyle="->", lw=1.2, ls=":"))
ax.annotate("", xy=(.5, .22), xytext=(.5, .42), arrowprops=dict(arrowstyle="->", lw=1.2, ls=":"))
ax.annotate("", xy=(.79, .5), xytext=(.60, .5),
            arrowprops=dict(arrowstyle="->", lw=1.6, color="#c62828", ls="--"))
ax.text(.70, .56, "SIN LLAVE", ha="center", fontsize=7, color="#c62828", weight="bold")
figura(fig, "P42", "Diagrama de integracion entre dominios", "relaciones")
guardar(rel, "matriz_integracion")

### Respuesta

Ninguna relación es directa. La aduanera-portuaria es agregada por mes; TRM, ONI y eventos son contextuales; el vínculo entre declaración y buque o terminal es NO VIABLE por ausencia de llave.

### Explicación

Esta es la pregunta que protege al proyecto de su error más probable. Sería tentador afirmar que las importaciones de un mes llegaron en tales buques o por tal terminal. No hay ninguna llave pública que lo permita, y construirla por inferencia produciría correspondencias falsas presentadas con apariencia de dato.

### Limitación

Una llave a nivel de evento existiría si la DIAN publicara el manifiesto de carga con identificador de buque, lo que no ocurre.

**Fuente:** Elaboración propia a partir del esquema de ambas fuentes · **Fecha de corte:** 2026-06


## 🟢 P43 — ¿Qué porcentaje de registros o periodos queda vinculado en cada integración?

**Estado:** ejecutada


In [ ]:
ma, mp = set(sa["mes"]), set(sp["mes"])
com = ma & mp
emb = pd.DataFrame([
    {"etapa": "meses aduaneros", "n": len(ma)},
    {"etapa": "meses portuarios", "n": len(mp)},
    {"etapa": "meses vinculados", "n": len(com)},
    {"etapa": "solo aduanas", "n": len(ma - mp)},
    {"etapa": "solo puerto", "n": len(mp - ma)},
])
mostrar(emb)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(emb["etapa"][::-1], emb["n"][::-1],
        color=["#a5673f", "#a5673f", "#2e7d32", "#31708e", "#6a8f3d"][::-1])
ax.set_xlabel("número de meses")
figura(fig, "P43", "Embudo de integracion entre dominios", "meses")
print(f"periodo común: {min(com):%Y-%m} a {max(com):%Y-%m}")
print(f"cobertura del dominio portuario: {len(com)/len(mp)*100:.1f} %")
guardar(emb, "embudo_integracion")

### Respuesta

101 meses vinculados de 173 aduaneros y 102 portuarios. El periodo común es 2018-01 a 2026-05, es decir el 99 % del dominio portuario y el 58 % del aduanero.

### Explicación

El único mes portuario que no se vincula es junio de 2026, porque la aduana todavía no lo publicó. Ese detalle es la evidencia empírica de que el puerto va por delante en el calendario de publicación, y anticipa la respuesta de P44.

### Limitación

La vinculación es de periodos, no de registros: no existe integración a nivel de evento.

**Fuente:** Cálculo propio sobre ambas series · **Fecha de corte:** 2026-06


## 🟢 P44 — ¿Los desfases de publicación y frecuencia introducen sesgos al comparar las fuentes?

**Estado:** ejecutada


In [ ]:
disp = pd.DataFrame([
    {"fuente": "DANE IMPO", "ultimo_mes": "2026-05", "rezago_dias": 45,
     "disponible_al_predecir_mes_siguiente": False},
    {"fuente": "Supertransporte", "ultimo_mes": "2026-06", "rezago_dias": 60,
     "disponible_al_predecir_mes_siguiente": False},
    {"fuente": "TRM", "ultimo_mes": "2026-06", "rezago_dias": 0,
     "disponible_al_predecir_mes_siguiente": True},
    {"fuente": "ONI", "ultimo_mes": "2026-06", "rezago_dias": 15,
     "disponible_al_predecir_mes_siguiente": True},
])
mostrar(disp)

fig, ax = plt.subplots(figsize=(9, 2.8))
col = ["#2e7d32" if d else "#c62828" for d in disp["disponible_al_predecir_mes_siguiente"]]
ax.barh(disp["fuente"], disp["rezago_dias"], color=col)
ax.set_xlabel("rezago de publicación (días)")
figura(fig, "P44", "Rezago de publicacion por fuente", "días")
guardar(disp, "calendario_disponibilidad")

### Respuesta

El puerto llega a junio de 2026 y la aduana a mayo. Sin embargo, **ninguna de las dos está disponible para predecir el mes en curso**: ambas publican con rezago superior a un mes.

### Explicación

Aquí está el hallazgo que impide un atajo tentador. Como el puerto publica antes, se podría pensar en usarlo como predictor adelantado del CIF. No funciona: al momento de pronosticar el CIF de un mes, tampoco existe el dato portuario de ese mes. El desfase entre ambas es de un mes, pero el rezago de ambas frente al presente es mayor. Ignorar esto produciría fuga operacional, que es el error que la línea base del proyecto identificó como el más peligroso.

### Limitación

Los rezagos declarados son los plazos normativos; el rezago observado puede variar.

**Fuente:** DANE, Superintendencia de Transporte, Banco de la República y NOAA · **Fecha de corte:** 2026-06


## 🟢 P45 — ¿Qué indicadores integrados aportan información adicional frente a revisar cada fuente por separado?

**Estado:** ejecutada


In [ ]:
val = pd.DataFrame([
    {"pregunta_del_usuario": "¿Subió el comercio o subieron los precios?",
     "con_una_fuente": "no distinguible", "con_la_vista_integrada": "sí: CIF vs toneladas",
     "aporta": True},
    {"pregunta_del_usuario": "¿El aumento de valor vino con más carga física?",
     "con_una_fuente": "no", "con_la_vista_integrada": "sí", "aporta": True},
    {"pregunta_del_usuario": "¿Qué terminal absorbió el cambio?",
     "con_una_fuente": "solo puerto", "con_la_vista_integrada": "sí, por sociedad portuaria",
     "aporta": True},
    {"pregunta_del_usuario": "¿Qué buque trajo esa mercancía?",
     "con_una_fuente": "no", "con_la_vista_integrada": "NO: sin llave verificable",
     "aporta": False},
])
mostrar(val)

j = integrado.copy()
j["cif_base100"] = j["cif_usd"] / j["cif_usd"].iloc[0] * 100
j["ton_base100"] = j["toneladas_totales"] / j["toneladas_totales"].iloc[0] * 100
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(j["mes"], j["cif_base100"], lw=1.3, label="valor CIF (aduana)")
ax.plot(j["mes"], j["ton_base100"], lw=1.3, label="toneladas movilizadas (puerto)")
ax.set_ylabel("índice base 100 = 2018-01"); ax.legend(fontsize=7)
figura(fig, "P45", "Valor economico frente a volumen fisico", "índice base 100")
guardar(val, "valor_incremental")

### Respuesta

Tres de cuatro preguntas del usuario solo se responden con la vista integrada. La cuarta sigue sin respuesta porque exigiría una llave que no existe.

### Explicación

La figura es el argumento central del proyecto integrado: el valor económico y el volumen físico se separan visiblemente a partir de cierto punto. Con una sola fuente ese hecho es invisible. Esa separación es la respuesta operativa a la pregunta de si un mes cambió por precio o por cantidad, y es lo que justifica el costo de integrar dos dominios en vez de quedarse con uno.

### Limitación

El valor incremental está argumentado sobre casos de uso, no medido con usuarios reales.

**Fuente:** Cálculo propio sobre la vista integrada · **Fecha de corte:** 2026-06


---

# Bloque 7 · Pronóstico, alertas y producto


## 🟢 P46 — ¿Qué indicadores tienen calidad, historia, frecuencia y utilidad suficientes para ser pronosticados?

**Estado:** ejecutada


In [ ]:
eleg = pd.DataFrame([
    {"indicador": "cif_usd", "dominio": "aduanero", "n_obs": len(sa), "continuidad": "completa",
     "rezago_dias": 45, "elegible": True},
    {"indicador": "peso_neto_kg", "dominio": "aduanero", "n_obs": len(sa),
     "continuidad": "completa", "rezago_dias": 45, "elegible": True},
    {"indicador": "toneladas_totales", "dominio": "portuario", "n_obs": len(sp),
     "continuidad": "completa", "rezago_dias": 60, "elegible": True},
    {"indicador": "ton_contenerizada", "dominio": "portuario", "n_obs": len(sp),
     "continuidad": "completa", "rezago_dias": 60, "elegible": True},
    {"indicador": "TEU", "dominio": "portuario", "n_obs": 0, "continuidad": "—",
     "rezago_dias": None, "elegible": False},
    {"indicador": "arribos", "dominio": "marítimo", "n_obs": 0, "continuidad": "—",
     "rezago_dias": None, "elegible": False},
    {"indicador": "permanencia_media", "dominio": "operacional", "n_obs": 0,
     "continuidad": "—", "rezago_dias": None, "elegible": False},
])
mostrar(eleg)

fig, ax = plt.subplots(figsize=(9, 3.2))
col = ["#2e7d32" if e else "#c62828" for e in eleg["elegible"]]
ax.barh(eleg["indicador"][::-1], eleg["n_obs"][::-1], color=col[::-1])
ax.axvline(36, ls="--", color="grey")
ax.text(37, .2, "mínimo para backtest", fontsize=7, color="grey")
ax.set_xlabel("observaciones mensuales disponibles")
figura(fig, "P46", "Elegibilidad de indicadores para pronostico", "número de observaciones")
guardar(eleg, "elegibilidad_indicadores")

### Respuesta

Cuatro indicadores elegibles de siete evaluados. Los tres descartados no tienen ni una sola observación porque su fuente no existe.

### Explicación

La regla se aplica antes de modelar, no después de ver los resultados. Un indicador sin historia suficiente no se pronostica aunque el usuario lo pida: se describe. Fijar el umbral en 36 observaciones antes de mirar los datos evita el sesgo de acomodar el criterio al indicador que uno quería incluir.

### Limitación

La utilidad para el usuario se evaluó por razonamiento, no con usuarios reales.

**Fuente:** Elaboración propia sobre las series construidas · **Fecha de corte:** 2026-06


## 🟢 P47 — ¿Qué desempeño obtienen Naive 1, Naive estacional y drift para cada indicador seleccionado?

**Estado:** ejecutada


In [ ]:
base = backtest_lineas_base(sp, ["toneladas_totales", "ton_contenerizada"], n_cortes=24)
mostrar(base)

fig, ax = plt.subplots(figsize=(9, 3.2))
piv = base.pivot(index="modelo", columns="objetivo", values="wape_pct")
piv.plot(kind="bar", ax=ax, color=["#31708e", "#a5673f"])
ax.set_ylabel("WAPE (%)"); ax.set_xlabel("")
plt.xticks(rotation=0)
figura(fig, "P47", "Desempeno de las lineas base por indicador", "WAPE en porcentaje")
guardar(base, "lineas_base_portuarias")

### Respuesta

Para toneladas totales, Naive 1 obtiene 9,17 % de WAPE y Naive 12 llega a 12,84 %. Para carga contenerizada, Naive 1 obtiene 8,55 %.

### Explicación

Las tres líneas base se reportan siempre, no solo la estacional. En la versión anterior del proyecto se reportaba únicamente Naive 12, y eso hacía parecer que el modelo mejoraba un 55 % cuando frente a Naive 1 la mejora real era del 8 %. La línea base más exigente es la que define si el modelo aporta algo.

### Limitación

Con 24 cortes, las diferencias menores a un punto porcentual deben leerse con cautela.

**Fuente:** Cálculo propio, backtest walk-forward de un paso · **Fecha de corte:** 2026-06


## 🟢 P48 — ¿Qué modelos logran mejor estabilidad fuera de muestra sin fuga?

**Estado:** ejecutada


In [ ]:
met = pd.read_csv(RUTA_METRICAS) if RUTA_METRICAS.exists() else metricas_portuarias
mostrar(met[met.ventana == 24][["objetivo", "modelo", "wape_pct", "mase_12"]].round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for i, obj in enumerate(["toneladas_totales", "ton_contenerizada"]):
    s = met[(met.ventana == 24) & (met.objetivo == obj)].sort_values("wape_pct")
    col = ["#2e7d32" if m not in ("naive_1", "naive_12", "drift") else "#9e9e9e"
           for m in s["modelo"]]
    ax[i].barh(s["modelo"][::-1], s["wape_pct"][::-1], color=col[::-1])
    ax[i].set_title(obj, fontsize=9); ax[i].set_xlabel("WAPE (%)")
figura(fig, "P48", "Comparacion de modelos frente a lineas base", "WAPE en porcentaje")
guardar(met, "metricas_modelos_portuarios")

### Respuesta

Para toneladas totales, Ridge obtiene 6,61 % contra 9,17 % de Naive 1: una mejora real del 28 %. Para carga contenerizada, **Naive 1 gana con 8,55 % y Ridge queda en 9,45 %**: el modelo es peor que repetir el último valor observado.

### Explicación

Este resultado dividido es el más valioso del bloque. Un indicador se puede pronosticar y el otro no, y ambos vienen de la misma fuente y el mismo pipeline. Reportar solo el caso exitoso habría sido más cómodo y menos cierto. Para la carga contenerizada la recomendación es usar Naive 1 y presentar el indicador de forma descriptiva.

### Limitación

Sin auditoría de fuga en verde ninguna de estas métricas sería publicable; la auditoría se ejecuta en el pipeline antes de calcularlas.

**Fuente:** Cálculo propio, backtest walk-forward con escalado dentro de cada corte · **Fecha de corte:** 2026-06


## 🟢 P49 — ¿Qué variables aportan valor incremental al pronóstico?

**Estado:** ejecutada


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

d = integrado.merge(ext_m, on="mes", how="left").sort_values("mes").reset_index(drop=True)
OBJ = "cif_usd"
# TODAS las variables se rezagan: al predecir el mes t no existe ningun dato de t
for k in (1, 2, 3, 12):
    d[f"cif_lag{k}"] = d[OBJ].shift(k)
for k in (1, 2, 3):
    d[f"ton_lag{k}"] = d["toneladas_totales"].shift(k)
    d[f"cont_lag{k}"] = d["ton_contenerizada"].shift(k)
    d[f"trm_lag{k}"] = d["trm_cop_usd"].shift(k)
d["cif_ma3"] = d[OBJ].shift(1).rolling(3).mean()
d["ton_ma3"] = d["toneladas_totales"].shift(1).rolling(3).mean()
d["mes_num"] = d["mes"].dt.month
d["tendencia"] = np.arange(len(d))
d["oni_lag1"] = d["oni_anomalia"].shift(1)

A = ["cif_lag1", "cif_lag2", "cif_lag3", "cif_lag12", "cif_ma3"]
CAL = ["mes_num", "tendencia"]
PUE = ["ton_lag1", "ton_lag2", "ton_lag3", "cont_lag1", "ton_ma3"]
CTX = ["trm_lag1", "trm_lag2", "trm_lag3", "oni_lag1"]
CONJ = {"A · historia propia": A, "B · A + calendario": A + CAL,
        "C · A + PUERTO": A + PUE, "D · A + contexto": A + CTX,
        "E · integrado completo": A + CAL + PUE + CTX}

dd = d.dropna().reset_index(drop=True)
N = 24
filas = []
for nombre, cols in CONJ.items():
    X, y = dd[cols].values, dd[OBJ].values
    ini = len(y) - N
    obs, pred = [], []
    for t in range(ini, len(y)):
        sc = StandardScaler().fit(X[:t])
        m = Ridge(alpha=1.0).fit(sc.transform(X[:t]), y[:t])
        pred.append(float(m.predict(sc.transform(X[[t]]))[0]))
        obs.append(float(y[t]))
    filas.append({"conjunto": nombre, "n_vars": len(cols), "wape_pct": round(wape(obs, pred), 3)})

y = dd[OBJ].values
ini = len(y) - N
for nom, f in [("naive_1", lambda h: h[-1]),
               ("naive_12", lambda h: h[-12] if len(h) >= 12 else h[-1]),
               ("drift", lambda h: h[-1] + (h[-1] - h[0]) / (len(h) - 1))]:
    obs = [y[t] for t in range(ini, len(y))]
    pred = [f(y[:t]) for t in range(ini, len(y))]
    filas.append({"conjunto": f"base · {nom}", "n_vars": 0,
                  "wape_pct": round(wape(obs, pred), 3)})

abl = pd.DataFrame(filas).sort_values("wape_pct").reset_index(drop=True)
base_a = abl.loc[abl.conjunto == "A · historia propia", "wape_pct"].iloc[0]
abl["ganancia_vs_A_pp"] = (base_a - abl["wape_pct"]).round(3)
mostrar(abl)

fig, ax = plt.subplots(figsize=(9.5, 3.4))
col = ["#2e7d32" if g > 0 else "#c62828" if "base" not in c else "#9e9e9e"
       for g, c in zip(abl["ganancia_vs_A_pp"], abl["conjunto"])]
ax.barh(abl["conjunto"][::-1], abl["wape_pct"][::-1], color=col[::-1])
ax.axvline(base_a, ls="--", color="#31708e")
ax.text(base_a + .05, -.4, "solo historia propia", fontsize=6.5, color="#31708e")
ax.set_xlabel("WAPE (%)  ·  menor es mejor")
figura(fig, "P49", "Ablacion multidominio sobre el pronostico del CIF", "WAPE en porcentaje",
       "2018-01 a 2026-05")

c = abl.loc[abl.conjunto == "C · A + PUERTO", "wape_pct"].iloc[0]
print(f"¿El puerto mejora el pronostico aduanero? {'SI' if c < base_a else 'NO'}")
print(f"  historia propia: {base_a:.3f} %   ->   con puerto: {c:.3f} %")
guardar(abl, "ablacion_multidominio")

### Respuesta

**Las variables portuarias NO mejoran el pronóstico del CIF: lo empeoran.** Solo la historia propia da 6,082 % de WAPE; añadiendo variables portuarias sube a 6,575 %. El modelo integrado completo (16 variables) tampoco mejora: 6,167 %. El único conjunto que aporta es historia propia más calendario, con 5,875 %. Las cinco configuraciones superan a las tres líneas base.

### Explicación

Este es el resultado más importante de la versión 5, y es negativo. La hipótesis implícita del proyecto integrado era que cruzar dominios mejoraría la predicción. Los datos dicen que no. La explicación tiene sentido: ambas fuentes miden el mismo comercio subyacente con el mismo rezago de publicación, así que el puerto no aporta información que la historia del propio CIF no contenga ya, y sí aporta ruido y grados de libertad consumidos sobre una muestra de 89 filas. **El valor de la integración no es predictivo: es explicativo.** Sirve para responder si un mes cambió por valor o por volumen físico, y para localizar por qué terminal pasa la carga. Presentarla como una mejora del pronóstico sería afirmar algo que este análisis refuta.

### Limitación

La ablación se hace sobre 89 filas utilizables y 24 cortes, con Ridge. Un conjunto de variables portuarias distinto o una muestra más larga podrían dar otro resultado, pero la conclusión debe reportarse tal como se midió.

**Fuente:** Cálculo propio sobre la vista integrada, backtest walk-forward de un paso · **Fecha de corte:** 2026-06


## 🟢 P50 — ¿Los modelos presentan sesgo, errores extremos o degradación por régimen?

**Estado:** ejecutada


In [ ]:
diag = met[met.ventana == 24][["objetivo", "modelo", "wape_pct", "sesgo_rel_pct",
                                "error_maximo"]].round(3)
mostrar(diag)

fig, ax = plt.subplots(figsize=(9, 3.2))
s = diag[diag.modelo == "ridge"]
ax.barh(s["objetivo"], s["sesgo_rel_pct"], color=["#31708e", "#a5673f"])
ax.axvline(0, color="grey", lw=.8)
ax.set_xlabel("sesgo relativo (%)  ·  positivo = el modelo subestima")
figura(fig, "P50", "Sesgo relativo por indicador", "porcentaje")
guardar(diag, "diagnostico_residuos")

### Respuesta

Se reporta sesgo relativo y error máximo por indicador y modelo. Un sesgo positivo significa que el modelo subestima de forma sistemática.

### Explicación

El sesgo importa más que el error promedio para un producto operativo. Un modelo que se equivoca poco pero siempre hacia el mismo lado induce decisiones consistentemente sesgadas, y eso es peor que un error mayor pero simétrico. El error máximo cumple otra función: un promedio aceptable puede esconder un mes catastrófico.

### Limitación

La degradación por régimen requiere más historia posterior al cambio de nivel de 2025.

**Fuente:** Cálculo propio sobre los residuos del backtest · **Fecha de corte:** 2026-06


## 🟢 P51 — ¿Los intervalos alcanzan la cobertura nominal y mantienen un ancho útil?

**Estado:** ejecutada


In [ ]:
cob = cobertura_intervalos_tabla
mostrar(cob)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(cob["caso"], cob["cobertura_empirica"] * 100, color="#31708e")
ax.axvline(80, ls="--", color="crimson", lw=1.6)
ax.text(81, -.4, "nominal 80 %", fontsize=7, color="crimson")
ax.set_xlabel("cobertura empírica (%)")
figura(fig, "P51", "Cobertura empirica frente al nivel nominal", "porcentaje")
guardar(cob, "cobertura_intervalos")

### Respuesta

Los intervalos se construyen con cuantiles empíricos de los errores fuera de muestra y calibración expansiva. La cobertura se mide, no se declara.

### Explicación

La regla que este bloque hace cumplir es que un intervalo nunca se deriva del WAPE ni de ninguna métrica de error puntual, que es exactamente lo que hacía la versión anterior del proyecto. Si la cobertura medida no alcanza el nivel declarado, el intervalo se recalibra o se renombra según lo que realmente cubre. Con pocos cortes evaluables, la incertidumbre sobre la propia cobertura es alta y eso también se declara.

### Limitación

Con menos de veinte cortes evaluables el intervalo de confianza de la cobertura es demasiado ancho para concluir.

**Fuente:** Cálculo propio, calibración conformal expansiva · **Fecha de corte:** 2026-06


## 🟢 P52 — ¿Cómo debe transformar el dashboard los indicadores en señales normales, de seguimiento o de alerta?

**Estado:** ejecutada


In [ ]:
matriz = pd.DataFrame([
    {"nivel": "normal", "criterio": "dentro del comportamiento histórico esperado",
     "accion": "seguimiento de rutina del informe mensual"},
    {"nivel": "seguimiento", "criterio": "variación relevante dentro del rango histórico",
     "accion": "revisar composición por país, capítulo o tipo de carga"},
    {"nivel": "alerta", "criterio": "desviación fuera del umbral o fuera del intervalo",
     "accion": "revisión dirigida y verificación de la fuente"},
])
mostrar(matriz)

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
for i, (c, n, t) in enumerate([("#2e7d32", "NORMAL", "rutina"),
                               ("#ef6c00", "SEGUIMIENTO", "revisar composición"),
                               ("#c62828", "ALERTA", "revisión dirigida")]):
    ax.add_patch(plt.Rectangle((i * .34, .3), .3, .4, fc=c, alpha=.85))
    ax.text(i * .34 + .15, .55, n, ha="center", color="white", fontsize=10, weight="bold")
    ax.text(i * .34 + .15, .4, t, ha="center", color="white", fontsize=7)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
figura(fig, "P52", "Matriz de decision de la senal de alerta", "niveles")
guardar(matriz, "matriz_decision")

### Respuesta

Tres niveles con criterio y acción sugerida. Los umbrales se calibran sobre la variación interanual y únicamente con la ventana de entrenamiento.

### Explicación

Hay una decisión de diseño detrás que vale explicar. La primera versión de estas reglas comparaba el pronóstico contra la mediana histórica del nivel. En una serie con tendencia, eso hace que todo mes reciente quede en el percentil 100 y la alerta se dispare siempre. Una alerta que suena siempre no informa nada. Por eso el estadístico se calcula sobre la variación interanual, que sí es estacionaria.

### Limitación

Cada alerta explica su razón y no constituye una orden operativa. Los umbrales no han sido validados con un usuario real.

**Fuente:** Elaboración propia, calibrada solo con la ventana de entrenamiento · **Fecha de corte:** 2026-06


---

# Cierre: matriz de trazabilidad


In [ ]:
resumen = pd.DataFrame([
    {"pregunta": p, "bloque": b, "estado": e}
    for p, b, e in ESTADOS])
mostrar(resumen, n=60)

conteo = resumen["estado"].value_counts()
fig, ax = plt.subplots(figsize=(8, 3))
col = {"ejecutada": "#2e7d32", "ejecutada parcialmente": "#f9a825",
       "no viable por ausencia de fuente": "#c62828",
       "no viable por cobertura insuficiente": "#8e24aa"}
ax.barh(conteo.index, conteo.values, color=[col.get(i, "#999") for i in conteo.index])
ax.set_xlabel("número de preguntas")
figura(fig, "CIERRE", "Estado de las 52 preguntas", "número de preguntas")
guardar(resumen, "matriz_trazabilidad_eda")

print(f"\nTotal: {len(resumen)} preguntas")
print(conteo.to_string())
print(f"\nSin responder ni justificar: {52 - len(resumen)}")
print(f"Figuras generadas: {len(list(FIGURAS.glob('*.png')))}")
print(f"Archivos de evidencia: {len(list(SALIDAS.glob('*.csv')))}")

---

## Conclusión del EDA

**Las 52 preguntas están respondidas.** No todas con un número: once están cerradas como
no viables, con la búsqueda documentada y la fuente que haría falta para abrirlas en una
fase futura. La especificación admite esa respuesta y exige que esté demostrada.

### Los tres hallazgos que definen el alcance del producto

1. **El dataset portuario es mensual y descargable por API**, no trimestral en PDF como
   se creía. Eso convirtió un dominio que parecía inviable en uno con 102 observaciones.

2. **El dominio marítimo y el operativo no existen como dato público.** Arribos, tipos de
   buque, ETA, ATA y permanencias no tienen serie histórica accesible. Documentarlo es más
   valioso que estimarlo.

3. **La integración es agregada por mes, nunca directa.** No hay llave pública entre una
   declaración de importación y un buque o una terminal, y construirla por inferencia
   produciría correspondencias falsas con apariencia de dato.

### Lo que el producto puede afirmar

Que el valor económico y el volumen físico del comercio de Buenaventura se han separado
en los últimos años, y que esa separación es medible cruzando dos fuentes oficiales.

### Lo que no puede afirmar

Nada sobre congestión, tiempos de atención, buques o terminales específicas asociadas a
una importación concreta.
